# CrossFuse v4 — Trainable Backbone, Matched-Estimator Calibration, Real Cross-Dataset Transfer

This is a from-the-ground-up rewrite of the MODEL and TRAINING pipeline, driven by a full audit of
`deepfake v3.3.ipynb`'s own outputs (its 8-row ablation grid, per-epoch train/val AUC logs, the
DFDC confusion matrix, and its calibration diagnostics). v3.3's evaluation SCAFFOLDING was sound —
identity-disjoint splitting, val/test separation, temperature scaling, bootstrap CIs, MC-Dropout
uncertainty — and is kept. Its RESULTS were not publishable as an accuracy paper, and both failures
traced to root causes visible in its own numbers, not to any single hyperparameter:

**Root cause 1 — frozen, globally-pooled ImageNet features.** v3.3 cached one 1792-d
globally-average-pooled EfficientNet-B4 vector per face and never trained anything below it. Eight
architecturally different ablation configs all landed inside a 0.03 AUC band (0.7508–0.8280): when
architecture stops mattering, the representation is the bottleneck. Train video AUC hit 0.9947
against val 0.7854 (a +0.247 gap), and the self-supervised sync head could not exceed 0.5145 AUC on
its OWN TRAINING DATA — global pooling had destroyed the mouth region entirely.

**Root cause 2 — calibration fit under one inference mode, applied under another.** v3.3 fit
temperature and both thresholds with dropout OFF, then evaluated test with 20-sample MC-Dropout
(dropout ON) averaging. That mismatch produced a video ECE of 0.19 AFTER "calibration", an
implausible fitted temperature of 4.3, and an audio threshold that needed an arbitrary
`min_recall_floor=0.5` patch to stop collapsing.

**What v4 changes:**
- **Cell 3 caches face CROPS, not embeddings** — pixels, not a frozen vector — so Cell 4 can
  fine-tune an EfficientNet-B0 backbone end-to-end with fresh augmentation every epoch.
- **Contiguous frames + exact timestamps**, so the sync head gets audio genuinely time-aligned to
  its visual window, instead of a whole-clip MFCC compared against whole-clip-strided frames.
- **Calibration (Cell 5) is fit on the SAME MC-Dropout estimator used at test** — the mismatch that
  produced Root Cause 2 cannot recur by construction.
- **No pre-baked augmented replicas** — self-blend and compression augmentation move to train time
  (Cell 4), removing the pseudo-replication that made every v3.3 bootstrap CI ~1.7× too narrow and
  let near-duplicates leak across the random-split ablation (A8).
- **Sample-proportional, category-balanced 60/20/20 splitting** (not v3.3's group-count split, which
  gave different-sized test sets to different ablation rows).
- **A new `fusion_head`** makes cross-attention load-bearing for the PRIMARY verdict, with a fair,
  equal-capacity late-fusion baseline (ablation A9) deciding whether the paper's title is earned.
- **Celeb-DF v2 replaces DFDC** as the cross-dataset headline (DFDC's attached sample gave a
  chance-inclusive CI); **ASVspoof 2019 LA** adds a cross-generator audio test the 0.997 in-dataset
  audio AUC had never faced.

`deepfake v3.4.ipynb` is NOT the basis for this rewrite — it reshuffled heads without touching either
root cause and inherited the same ceiling. This notebook is built from v3.3 forward.

**Kaggle execution order (three sessions — see the project plan for full detail):**
1. **Stage A** (Cells 1, 3): cache crops + waveforms, ~1,500 clips. Publish `/kaggle/working/crops_v4`
   as a Kaggle Dataset before Stage B.
2. **Stage B, main** (Cells 1, 2, 4, 5, 6, 6b): attach the dataset, train 3 seeds, calibrate, report,
   run Celeb-DF + ASVspoof.
3. **Stage B, ablations** (Cells 1, 2, 4, 8, 9): A1–A10, 2 seeds each, then export.

### Cell 1 — Environment, CONFIG & Shared Helpers
Everything downstream reads from one `CONFIG` dict (also what Cell 8's ablation runner mutates per
row). New in v4: `extract_face_crops_contiguous()` replaces the frozen-embedding extractor entirely
(this is the Root-Cause-1 fix), plus a window-aligned `logmel_window()` front-end for the sync head.
`UnionFind`/identity-token resolution moved up from Cell 3 so Cell 6 (Celeb-DF) can reuse it without
depending on Stage A having run in the same session. `ffmpeg_extract_wav`, `make_rng`, and CMVN are
retained verbatim from v3.3 — they were correct.

In [ ]:
# =====================================================================
# CELL 1: ENVIRONMENT, CONFIG & SHARED HELPERS  (v4.0)
#
# WHAT CHANGED vs v3.3 -- and WHY. v3.3's own ablation grid showed eight
# architecturally different configurations all landing inside a 0.03 band
# (video AUC 0.7508-0.8280). When architecture stops mattering, the
# REPRESENTATION is the bottleneck. v3.3 cached the output of
# `efficientnet_b4(weights=IMAGENET); classifier = Identity()` -- one
# 1792-d GLOBALLY-AVERAGE-POOLED vector per face, with nothing below it
# ever trained. That representation encodes "who this face is" (ImageNet
# embeddings are excellent at that), which is why train video AUC hit
# 0.9947 against val 0.7854, and why the sync head could not reach even
# 0.52 AUC ON ITS OWN TRAINING DATA: global pooling destroys the mouth
# region, so nothing correlated with speech survives.
#
#   1. WE CACHE FACE CROPS, NOT EMBEDDINGS.
#      extract_face_crops_contiguous() replaces
#      extract_visual_embeddings_batched(). It returns raw uint8 160x160
#      crops, so Cell 4 can fine-tune a backbone end-to-end and apply
#      FRESH image-level augmentation every epoch. This is the single
#      change the whole of v4 rests on.
#
#   2. FRAMES ARE CONTIGUOUS, NOT STRIDED ACROSS THE WHOLE CLIP.
#      v3.3 sampled 60 frames at `total_frames // 60` stride while
#      interpolating MFCCs to a fixed 128 over the same whole clip -- so
#      audio and video shared no common time base at all. AV-sync needs
#      sub-100ms correspondence. We now cache a CONTIGUOUS 64-frame run
#      (~2.1-2.6s at native fps) and record its exact timestamps, so
#      Cell 4 can slice audio to precisely the window the frames cover.
#
#   3. NO PRE-BAKED AUGMENTED REPLICAS.
#      v3.3 stored 2 compression replicas + 1 self-blend per clip as
#      SEPARATE samples. That turned ~1,500 independent clips into 5,000
#      "samples", so every bootstrap CI treated ~3x more independent
#      observations than actually existed (CIs came out ~1.7x too narrow)
#      and the random-split ablation A8 leaked near-duplicates across
#      splits on top of identities. Augmentation moves to TRAIN TIME
#      (Cell 4), where it is also strictly stronger -- a fresh random
#      draw each epoch instead of two frozen copies.
#
#   4. UnionFind / get_identity_tokens / build_identity_groups MOVED HERE
#      from Cell 3, so Cell 6 (Celeb-DF) can reuse the same grouping
#      logic without depending on Stage A having run in this session.
#
# RETAINED VERBATIM from v3.3 (these were correct): seed_everything,
# make_rng per-tag RNG streams, ffmpeg_extract_wav as THE single audio
# decode path, cmvn_normalize/maybe_cmvn.
# =====================================================================
import os
import sys
import glob
import json
import math
import random
import hashlib
import warnings
import subprocess
import tempfile

warnings.filterwarnings("ignore", category=UserWarning, module="librosa")
warnings.filterwarnings("ignore", category=FutureWarning, module="librosa")
warnings.filterwarnings("ignore", category=UserWarning, module="torchvision")

print("Installing extra dependencies WITHOUT touching numpy/torch/torchvision...")
# --no-deps => pip installs ONLY these packages' own code and resolves
# nothing else, so it is impossible for numpy to be changed here.
!pip install -q --no-deps facenet-pytorch librosa ffmpeg-python tqdm soundfile audioread pooch lazy_loader
print("Installation complete!")

import PIL._util
if not hasattr(PIL._util, 'is_directory'):
    PIL._util.is_directory = os.path.isdir
if not hasattr(PIL._util, 'is_path'):
    PIL._util.is_path = lambda x: isinstance(x, (str, bytes, os.PathLike))

import re
import cv2
import librosa
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from facenet_pytorch import MTCNN
from sklearn.metrics import (roc_auc_score, accuracy_score, balanced_accuracy_score,
                             confusion_matrix, roc_curve, auc, classification_report)
import matplotlib.pyplot as plt
import seaborn as sns


def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(seed=42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpus = torch.cuda.device_count()
print(f"Environment ready! Hardware acceleration: {device.type.upper()} "
      f"({n_gpus} GPU{'s' if n_gpus != 1 else ''} visible"
      f"{': ' + ', '.join(torch.cuda.get_device_name(i) for i in range(n_gpus)) if n_gpus else ''})")
print(f"Version check -> numpy: {np.__version__} | torch: {torch.__version__} | "
      f"torchvision: {torchvision.__version__}")

# ---- v4 CONFIG ---------------------------------------------------------
# One place the Cell 8 ablation runner mutates; every other cell reads
# from this dict instead of hardcoding its own flags.
CONFIG = {
    # ---- Stage A: what gets cached ----
    "CROP_SIZE":               160,   # face crop edge, px (uint8 on disk)
    "CACHED_FRAMES":           64,    # CONTIGUOUS frames cached per clip
    "FACE_MARGIN":             0.30,  # box expansion, fraction of box size
    "AUDIO_SR":                16000,
    "CAP_FVRA":                250,   # category caps, retained from v3.3
    "CAP_FVFA":                250,

    # ---- Stage B: what the model sees ----
    "WINDOW_FRAMES":           32,    # contiguous sub-window per training sample
    "AUDIO_FRAMES":            128,   # full-clip MFCC length (audio-forgery head)
    "SYNC_MEL_FRAMES":         128,   # window-aligned log-mel length (sync head)
    "N_MELS":                  80,
    "BACKBONE":                "efficientnet_b0",
    "FREEZE_BLOCKS":           4,     # features[:4] stay frozen (BN in eval)

    # ---- Ablation toggles (Cell 8 flips these) ----
    "GENERATE_HARD_NEGATIVES": True,  # on-the-fly self-blend, TRAIN SPLIT ONLY
    "USE_COMPRESSION_AUG":     True,  # train-time JPEG/downscale/noise
    "USE_MFCC_CMVN":           True,
    "USE_CROSS_ATTENTION":     True,
    "USE_SYNC_HEAD":           True,  # requires USE_CROSS_ATTENTION
    "USE_FUSION_HEAD":         True,  # CA-fused overall verdict (A9 compares vs late fusion)
    "LATE_FUSION_BASELINE":    False, # A9: equal-capacity MLP on concat[v_rep,a_rep], no CA
    "SYNC_DENSE_ALIGNED":      True,  # A10: False = v3.3-style whole-clip-strided sync input
    "IDENTITY_DISJOINT_SPLIT": True,  # False = ablation A8 (random split, by clip)
    "MODALITY":                "both",
    "HELD_OUT_CATEGORY":       None,

    # ---- Splits, training, evaluation ----
    "SPLIT_FRACS":             (0.60, 0.20, 0.20),   # by SAMPLE count, not group count
    "EPOCHS_FROZEN":           5,
    "EPOCHS_FINETUNE":         25,
    "PATIENCE":                7,
    "BATCH_CLIPS":             4,
    "GRAD_ACCUM":              2,
    "LR_HEAD":                 1e-4,
    "LR_BACKBONE":             3e-5,
    "LLRD":                    0.75,  # layer-wise LR decay across backbone blocks
    "N_MC_SAMPLES":            20,
    "N_SEEDS":                 3,
    "SEED":                    42,
    "SYNC_MIN_AUC":            0.70,  # PREREGISTERED: below this the sync head is
                                      # not reported as a verdict signal anywhere.
    "NUM_WORKERS":             2,
}
print("CONFIG ->", json.dumps({k: v for k, v in CONFIG.items()}, indent=2, default=str))

N_DEVICES = max(1, torch.cuda.device_count())
DEVICE_IDS = [f"cuda:{i}" for i in range(N_DEVICES)] if torch.cuda.is_available() else ["cpu"]
print(f"Detected extraction devices: {DEVICE_IDS}")

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


# ---- Per-tag deterministic RNG streams (retained from v3.3) -------------
# Each independent random decision gets its OWN seeded stream keyed by a
# string tag, instead of every cell sharing one global `random` state whose
# position depends on how many earlier random calls already ran.
def make_rng(tag, seed=None):
    seed = CONFIG["SEED"] if seed is None else seed
    h = int(hashlib.md5(f"{tag}:{seed}".encode()).hexdigest(), 16) % (2**32)
    return random.Random(h)


# ---- Shared audio front-end (retained from v3.3) ------------------------
# THE single decode path for every clip the model ever sees -- training
# tensor, Celeb-DF clip, ASVspoof utterance, or user upload. Demuxing
# through ffmpeg first means librosa always reads a clean WAV via the fast
# soundfile backend; it can no longer silently drop to audioread mid-run
# (which is what produced v3.2's in-the-wild false-positive audio verdict).
def ffmpeg_extract_wav(video_path, sr=None):
    """Returns (y, sr) exactly like librosa.load, but demuxes through
    ffmpeg first so every caller shares one decode path. Falls back to
    direct librosa.load (with a LOUD warning, not a silent one) only if
    ffmpeg itself is unavailable or fails."""
    sr = CONFIG["AUDIO_SR"] if sr is None else sr
    with tempfile.TemporaryDirectory() as tmpdir:
        wav_path = os.path.join(tmpdir, "audio.wav")
        try:
            result = subprocess.run(
                ["ffmpeg", "-y", "-i", video_path, "-vn", "-ac", "1",
                 "-ar", str(sr), "-f", "wav", wav_path],
                stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, timeout=60)
            if result.returncode != 0 or not os.path.exists(wav_path):
                raise RuntimeError(result.stderr.decode(errors="ignore")[-300:])
            y, sr_out = librosa.load(wav_path, sr=sr)
            return y, sr_out
        except Exception as e:
            print(f"[!] ffmpeg_extract_wav: ffmpeg path failed for "
                  f"'{os.path.basename(video_path)}' ({e}); "
                  f"falling back to direct librosa.load (backend may differ!)")
            return librosa.load(video_path, sr=sr)


# ---- MFCC + CMVN for the audio-FORGERY head (retained from v3.3) --------
# This path reached 0.997 test AUC in v3.3 and is deliberately unchanged.
# CMVN is applied at LOAD TIME, never baked into the cache, so ablation A7
# (USE_MFCC_CMVN=False) stays a config toggle instead of a re-extraction.
def cmvn_normalize(mfcc, eps=1e-8):
    mfcc = np.asarray(mfcc, dtype=np.float32)
    mean = mfcc.mean(axis=1, keepdims=True)
    std = mfcc.std(axis=1, keepdims=True)
    return (mfcc - mean) / (std + eps)


def maybe_cmvn(mfcc, use_cmvn=None):
    use_cmvn = CONFIG["USE_MFCC_CMVN"] if use_cmvn is None else use_cmvn
    return cmvn_normalize(mfcc) if use_cmvn else np.asarray(mfcc, dtype=np.float32)


def mfcc_from_waveform(y, sr=None, n_frames=None):
    """Full-clip RAW (not CMVN'd) MFCC-40, time-interpolated to a fixed
    length. Split out from extract_audio_mfcc so Cell 4 can call it on the
    CACHED waveform without re-decoding the video."""
    sr = CONFIG["AUDIO_SR"] if sr is None else sr
    n_frames = CONFIG["AUDIO_FRAMES"] if n_frames is None else n_frames
    try:
        if y is None or len(y) < sr // 10:
            raise ValueError("audio too short or empty")
        mfcc = librosa.feature.mfcc(y=np.asarray(y, dtype=np.float32), sr=sr, n_mfcc=40)
        mfcc_t = torch.tensor(mfcc, dtype=torch.float32).unsqueeze(0)      # (1, 40, T_native)
        mfcc_t = F.interpolate(mfcc_t, size=n_frames, mode="linear", align_corners=False)
        return mfcc_t.squeeze(0).numpy().astype(np.float32)
    except Exception:
        return np.zeros((40, n_frames), dtype=np.float32)


def extract_audio_mfcc(video_path):
    """Decode-and-MFCC convenience wrapper for cells that start from a
    video path (Cells 6/6b/7)."""
    y, sr = ffmpeg_extract_wav(video_path)
    return mfcc_from_waveform(y, sr)


# ---- Window-aligned log-mel for the AV-SYNC head (NEW in v4) ------------
def logmel_window(y, sr, t0, t1, n_mels=None, out_frames=None):
    """Log-mel spectrogram of EXACTLY the [t0, t1) seconds of `y`, resampled
    along time to a fixed `out_frames`.

    This is the piece v3.3 did not have. Its sync head compared a
    whole-clip-strided visual sequence against a whole-clip MFCC, so the
    two streams shared no common time base and the correspondence task was
    unlearnable by construction (train sync AUC 0.5145 -- the model could
    not even memorise it in-sample). Here the caller passes the exact
    timestamps of the cached visual window, so mel frame k and visual frame
    j refer to the same instant, up to the fixed-length resample which
    preserves RELATIVE timing -- the only thing lip-sync depends on.

    hop_length=160 at 16 kHz = 10 ms/frame (100 fps), n_fft=400 = 25 ms:
    standard speech analysis framing, ~4x the video frame rate.
    """
    n_mels = CONFIG["N_MELS"] if n_mels is None else n_mels
    out_frames = CONFIG["SYNC_MEL_FRAMES"] if out_frames is None else out_frames
    try:
        y = np.asarray(y, dtype=np.float32)
        i0, i1 = max(0, int(t0 * sr)), min(len(y), int(t1 * sr))
        seg = y[i0:i1]
        if len(seg) < int(0.05 * sr):
            raise ValueError("window too short")
        mel = librosa.feature.melspectrogram(y=seg, sr=sr, n_fft=400, hop_length=160, n_mels=n_mels)
        logmel = librosa.power_to_db(mel, ref=np.max)
        t = torch.tensor(logmel, dtype=torch.float32).unsqueeze(0)         # (1, n_mels, T)
        t = F.interpolate(t, size=out_frames, mode="linear", align_corners=False)
        arr = t.squeeze(0).numpy()
        return ((arr - arr.mean()) / (arr.std() + 1e-8)).astype(np.float32)
    except Exception:
        return np.zeros((n_mels, out_frames), dtype=np.float32)


# ---- Hardened identity grouping (MOVED UP from v3.3 Cell 3) -------------
# Kept verbatim -- this was correct and is what makes the leakage assert in
# Cell 4 catch SOURCE-identity leakage, not just target-folder leakage.
class UnionFind:
    def __init__(self):
        self.parent = {}

    def find(self, x):
        self.parent.setdefault(x, x)
        root = x
        while self.parent[root] != root:
            root = self.parent[root]
        while self.parent[x] != root:
            self.parent[x], x = root, self.parent[x]
        return root

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[ra] = rb


def get_identity_tokens(video_path):
    """FakeAVCeleb layout: .../{category}/{ethnicity}/{gender}/{id}/{video}.mp4,
    but manipulated filenames also encode a SOURCE identity
    (e.g. id00076_id00812_wavtolip.mp4). Pull every id\\d+ token from both
    the parent folder AND the filename."""
    path_norm = video_path.replace("\\", "/")
    parent = os.path.basename(os.path.dirname(path_norm))
    fname = os.path.basename(path_norm)
    tokens = set(re.findall(r'id\d+', parent.lower())) | set(re.findall(r'id\d+', fname.lower()))
    if not tokens:
        tokens = {f"folder::{parent.lower()}"}
    return tokens


def build_identity_groups(video_paths):
    """Union every id-token that co-occurs on any single clip into one
    connected component; returns {video_path: canonical_group_id}."""
    uf = UnionFind()
    path_tokens = {}
    for v in video_paths:
        toks = get_identity_tokens(v)
        path_tokens[v] = toks
        toks = list(toks)
        for t in toks[1:]:
            uf.union(toks[0], t)
        if len(toks) == 1:
            uf.find(toks[0])
    return {v: uf.find(next(iter(path_tokens[v]))) for v in video_paths}


# ---- Face-crop extraction (NEW in v4 -- replaces the embedding path) ----
def build_detector(device_str):
    """Detector only. v3.3 also built an EfficientNet-B4 here to bake
    frozen embeddings into the cache; v4 caches pixels instead and trains
    its backbone in Cell 4, so Stage A needs nothing but MTCNN."""
    return MTCNN(image_size=CONFIG["CROP_SIZE"], margin=0, keep_all=False,
                 select_largest=True, post_process=False, device=device_str)


def _crop_from_box(pil_img, box, out_size, margin_frac):
    """Square, margin-expanded crop around an MTCNN box, clamped to the
    frame. Square-first (rather than resizing a rectangle) keeps the face
    aspect ratio undistorted, which matters when the artifact you are
    trying to detect is a blending boundary."""
    W, H = pil_img.size
    x1, y1, x2, y2 = box
    cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
    side = max(x2 - x1, y2 - y1) * (1.0 + margin_frac)
    half = side / 2.0
    l, t = int(round(cx - half)), int(round(cy - half))
    r, b = int(round(cx + half)), int(round(cy + half))
    l, t = max(0, l), max(0, t)
    r, b = min(W, r), min(H, b)
    if r - l < 8 or b - t < 8:
        return None
    return pil_img.crop((l, t, r, b)).resize((out_size, out_size), Image.BILINEAR)


def extract_face_crops_contiguous(video_path, mtcnn_local, rng=None,
                                  n_frames=None, out_size=None, batch_size=32):
    """Cache a CONTIGUOUS run of `n_frames` face crops as uint8.

    Returns (crops uint8 (n_frames, S, S, 3), valid bool (n_frames,), fps,
    start_frame, timestamps float32 (n_frames,)) or (None, None, 0.0, 0,
    None) on failure.

    `valid` is PER FRAME, not a count: Cell 4 needs to know WHICH frames
    carry a genuine detection so its attention mask can exclude the rest.
    v3.3 stored only a scalar n_valid_frames and had to assume the valid
    frames were a prefix, which they are not.

    Two deliberate differences from v3.3's extract_visual_embeddings_batched:

      * CONTIGUITY. v3.3 strode across the whole clip at total//60, so
        consecutive "frames" were ~0.2-0.5s apart and carried no usable
        motion or speech correspondence. We take a single unbroken run at
        native fps from a seeded random offset, and return its exact
        timestamps so Cell 4 can cut the matching audio window.
      * NO EMBEDDING. We return pixels. Everything downstream of the crop
        is trainable in v4.

    Frames where detection fails inherit the previous valid box (faces do
    not teleport between adjacent frames) and are marked invalid in the
    returned count, so the attention mask can exclude them.
    """
    n_frames = CONFIG["CACHED_FRAMES"] if n_frames is None else n_frames
    out_size = CONFIG["CROP_SIZE"] if out_size is None else out_size
    rng = rng or make_rng(f"crops:{os.path.basename(video_path)}")

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None, None, 0.0, 0, None
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = float(cap.get(cv2.CAP_PROP_FPS)) or 25.0
    if not np.isfinite(fps) or fps <= 1.0:
        fps = 25.0

    start = rng.randint(0, max(0, total - n_frames)) if total > n_frames else 0
    if start > 0:
        cap.set(cv2.CAP_PROP_POS_FRAMES, start)

    pil_frames, idxs = [], []
    fi = start
    while cap.isOpened() and len(pil_frames) < n_frames:
        ret, frame = cap.read()
        if not ret:
            break
        pil_frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
        idxs.append(fi)
        fi += 1
    cap.release()
    if len(pil_frames) == 0:
        return None, None, fps, start, None

    # Batched detection across the contiguous run.
    boxes_all = []
    for i in range(0, len(pil_frames), batch_size):
        chunk = pil_frames[i:i + batch_size]
        try:
            boxes, probs = mtcnn_local.detect(chunk)
        except Exception:
            boxes = [None] * len(chunk)
        if boxes is None:
            boxes = [None] * len(chunk)
        for b in boxes:
            boxes_all.append(None if b is None or len(b) == 0 else b[0])

    crops = np.zeros((n_frames, out_size, out_size, 3), dtype=np.uint8)
    valid = np.zeros(n_frames, dtype=bool)
    last_box = None
    for j, pil_img in enumerate(pil_frames):
        box = boxes_all[j] if j < len(boxes_all) else None
        detected = box is not None
        if not detected:
            box = last_box                      # faces don't teleport frame-to-frame
        crop = _crop_from_box(pil_img, box, out_size, CONFIG["FACE_MARGIN"]) if box is not None else None
        if crop is None:
            W, H = pil_img.size                 # last resort: centre square
            s = min(W, H)
            crop = pil_img.crop(((W - s) // 2, (H - s) // 2, (W + s) // 2, (H + s) // 2)) \
                          .resize((out_size, out_size), Image.BILINEAR)
        else:
            if detected:
                last_box = box
            valid[j] = True
        crops[j] = np.asarray(crop, dtype=np.uint8)

    n_real_frames = len(pil_frames)
    if n_real_frames < n_frames:
        valid[n_real_frames:] = False           # short clip: padding tail is never valid

    timestamps = np.zeros(n_frames, dtype=np.float32)
    for j in range(len(idxs)):
        timestamps[j] = idxs[j] / fps
    for j in range(len(idxs), n_frames):        # short clip: pad timestamps forward
        timestamps[j] = timestamps[max(0, len(idxs) - 1)] + (j - len(idxs) + 1) / fps

    return crops, valid, fps, start, timestamps


print("\nCell 1 complete (v4): CONFIG, seeding, ffmpeg audio front-end, MFCC/log-mel front-ends, "
      "identity grouping and the CONTIGUOUS FACE-CROP extractor are ready.")

### Cell 2 — CrossFuse v4 Architecture: Trainable Backbone, Spatially-Resolved Features,
### Fair Cross-Attention Ablation
`video_head` and `audio_head` still read ONLY their own branch (unchanged design intent from v3.3).
New: the EfficientNet-B0 backbone lives inside the model and is fine-tuned (`set_backbone_trainable`,
layer-wise LR decay via `backbone_param_groups`); backbone features keep a 3×3 SPATIAL grid instead of
being globally pooled, so the sync head gets a genuine mouth-region token from the bottom row instead
of a pooled vector with the mouth already averaged away. A new `fusion_head` reads the cross-attended
representation; `LATE_FUSION_BASELINE=True` swaps in an equal-capacity concatenation MLP with every
other architectural choice held fixed — this exact swap is ablation A9, the row that decides the
paper's title. `aux_head` (never reported in v3.3) is deleted.

In [ ]:
# =====================================================================
# CELL 2: CROSSFUSE v4 ARCHITECTURE -- TRAINABLE BACKBONE, SPATIALLY
#         RESOLVED FEATURES, AND A CROSS-ATTENTION PATH THAT CAN BE
#         FAIRLY ABLATED
#
# WHAT CHANGED vs v3.3, and why each change exists:
#
#  1. THE BACKBONE IS PART OF THE MODEL AND IS TRAINED.
#     v3.3's model started at a frozen 1792-d ImageNet vector. Its eight
#     ablation configurations then landed inside a 0.03 AUC band, which is
#     what "the representation, not the architecture, is the bottleneck"
#     looks like in a results table. Here EfficientNet-B0 lives inside the
#     module; Cell 4 freezes features[:FREEZE_BLOCKS] and fine-tunes the
#     rest with layer-wise LR decay.
#
#  2. FEATURES KEEP SPATIAL STRUCTURE.
#     The backbone map is pooled to (1280, 3, 3), not to a single vector.
#     Two consequences: the video head still gets a global token (mean over
#     the grid, i.e. what v3.3 had, but LEARNED), and the sync head gets a
#     separate MOUTH token from the bottom row of the grid. v3.3's global
#     average pool erased the mouth region entirely, which is the concrete
#     reason its sync head could not exceed 0.5145 AUC on its own TRAINING
#     data -- there was no mouth information left to correlate with speech.
#
#  3. THE SYNC HEAD READS A TIME-ALIGNED SPECTROGRAM.
#     `sync_audio` is the log-mel of exactly the seconds the visual window
#     covers (Cell 1's logmel_window). v3.3 compared whole-clip-strided
#     visual frames against a whole-clip MFCC -- no shared time base at
#     all. Ablation A10 sets SYNC_DENSE_ALIGNED=False to feed the v3.3-style
#     unaligned input through this same head, which isolates whether
#     alignment (not architecture) was the thing that killed it.
#
#  4. FUSION vs LATE FUSION IS AN EXACT, EQUAL-CAPACITY SWAP.
#     `fusion_head` reads a 4*hidden vector. With cross-attention that
#     vector is concat[pooled fused-V, pooled fused-A]; with
#     LATE_FUSION_BASELINE=True it is concat[v_rep, a_rep] -- also 4*hidden,
#     through an IDENTICALLY SHAPED MLP. So ablation A9 varies exactly one
#     thing: whether the representation passed through cross-attention.
#     v3.4's A9 instead compared against max() of two temperature-scaled
#     head probabilities, which is not invariant to per-head temperature;
#     its "null result" was an artefact of the comparison, not a finding.
#
#  5. `aux_head` IS DELETED. It was trained on OR-labels every step in
#     v3.3 and never appeared in any reported number.
#
#  6. THE VISUAL ENCODER IS SPLIT OUT (encode_visual / forward_from_tokens).
#     MC-Dropout needs N stochastic passes, but only the HEADS contain
#     dropout -- the backbone is deterministic. Running the backbone once
#     and only re-running the heads makes 20-sample MC-Dropout roughly
#     20x cheaper, which is what makes it affordable at all now that the
#     backbone is a real CNN over 32 frames instead of a cached vector.
#
# RETAINED VERBATIM from v3.3: masked_mean_std (padding-aware mean||std
# pooling), TemporalConvNet, the two independent attribution heads reading
# ONLY their own modality, and enable_mc_dropout's "dropout on, BatchNorm
# frozen" semantics.
# =====================================================================

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights


class TemporalConvNet(nn.Module):
    """Dilated temporal CNN over a (B, C_in, T) sequence -> (B, T_out, 256).
    Retained from v3.3, with num_inputs/target_seq_len now parameterised so
    the same block serves the 40-band MFCC (audio-forgery head) and the
    80-band log-mel (sync head)."""

    def __init__(self, num_inputs=40, num_channels=(64, 128, 256), kernel_size=3,
                 dropout=0.3, target_seq_len=60, out_dim=256):
        super().__init__()
        layers = []
        for i, out_channels in enumerate(num_channels):
            dilation_size = 2 ** i
            in_channels = num_inputs if i == 0 else num_channels[i - 1]
            layers += [
                nn.Conv1d(in_channels, out_channels, kernel_size,
                          padding=(kernel_size - 1) * dilation_size, dilation=dilation_size),
                nn.BatchNorm1d(out_channels),
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
        self.network = nn.Sequential(*layers)
        self.temporal_pool = nn.AdaptiveAvgPool1d(target_seq_len)
        self.proj = nn.Linear(num_channels[-1], out_dim)

    def forward(self, x):
        out = self.network(x)
        out = self.temporal_pool(out)
        out = out.transpose(1, 2)
        return self.proj(out)


def masked_mean_std(x, mask=None, eps=1e-8):
    """x: (B, T, D). mask: (B, T) bool, True = valid position (None = all
    valid). Returns concat[mean, std] -> (B, 2*D).

    Retained from v3.3. The std half is what carries frame-to-frame
    INCONSISTENCY -- the signal a plain mean throws away, and the one that
    matters most for manipulated video."""
    if mask is None:
        mean = x.mean(dim=1)
        var = x.var(dim=1, unbiased=False)
    else:
        mask_f = mask.unsqueeze(-1).float()
        count = mask_f.sum(dim=1).clamp(min=1.0)
        mean = (x * mask_f).sum(dim=1) / count
        var = ((x - mean.unsqueeze(1)) ** 2 * mask_f).sum(dim=1) / count
    std = torch.sqrt(var.clamp(min=eps))
    return torch.cat([mean, std], dim=-1)


def _mlp_head(in_dim, hidden, dropout):
    return nn.Sequential(
        nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(hidden, 1))


class CrossFuseModelV4(nn.Module):
    """
    Heads
    -----
    video_head   reads ONLY the visual branch  -> P(visual track manipulated)
    audio_head   reads ONLY the audio branch   -> P(audio track synthetic)
    sync_head    reads mouth<->log-mel cross-attention -> P(AV mismatch)
    fusion_head  reads the fused (or, for the A9 baseline, concatenated)
                 representation -> P(clip manipulated at all)

    The two attribution heads remain strictly unimodal, exactly as in v3.3:
    cross-attention cannot leak audio evidence into the video verdict or
    vice versa, which is what keeps the 4-way modality-attribution matrix
    interpretable.

    forward(crops, mfcc, sync_audio, visual_mask) -> dict with keys
    'video_logit', 'audio_logit', 'sync_logit', 'fusion_logit',
    'attn_v2a', 'attn_a2v'. All logits are RAW (train with
    BCEWithLogitsLoss, read with torch.sigmoid). sync_logit is None when
    use_sync_head is off; fusion_logit is None when use_fusion_head is off.
    """

    def __init__(self, hidden_dim=256, num_heads=4, dropout=0.4,
                 window_frames=32, sync_mel_frames=128, n_mels=80,
                 use_cross_attention=True, use_sync_head=True, use_fusion_head=True,
                 late_fusion_baseline=False, sync_dense_aligned=True,
                 freeze_blocks=4, pretrained=True):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.use_cross_attention = use_cross_attention
        self.use_sync_head = use_sync_head and use_cross_attention
        self.use_fusion_head = use_fusion_head
        self.late_fusion_baseline = late_fusion_baseline
        self.sync_dense_aligned = sync_dense_aligned
        self.freeze_blocks = freeze_blocks

        # ---- Trainable visual backbone -------------------------------
        weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
        self.backbone = efficientnet_b0(weights=weights).features   # -> (B, 1280, H/32, W/32)
        self.backbone_dim = 1280
        self.spatial_pool = nn.AdaptiveAvgPool2d(3)                 # -> (B, 1280, 3, 3)
        self.register_buffer("img_mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("img_std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        self.set_backbone_trainable(freeze_blocks)

        # ---- Visual branch (feeds video_head ONLY) -------------------
        self.visual_proj = nn.Linear(self.backbone_dim, hidden_dim)
        self.visual_encoder = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=num_heads, dim_feedforward=512,
            dropout=dropout, batch_first=True)
        self.video_head = _mlp_head(2 * hidden_dim, 128, dropout)

        # ---- Audio branch (feeds audio_head ONLY) --------------------
        # Unchanged from v3.3: full-clip MFCC-40 + CMVN -> TCN. This path
        # reached 0.997 test AUC and is deliberately left alone.
        self.audio_encoder = TemporalConvNet(num_inputs=40, target_seq_len=window_frames,
                                             dropout=0.3, out_dim=hidden_dim)
        self.audio_head = _mlp_head(2 * hidden_dim, 128, dropout)

        # ---- Sync branch: mouth tokens <-> time-aligned log-mel -------
        if self.use_sync_head:
            self.mouth_proj = nn.Linear(self.backbone_dim, hidden_dim)
            sync_in = n_mels if sync_dense_aligned else 40      # A10 feeds the v3.3-style MFCC
            self.mel_encoder = TemporalConvNet(num_inputs=sync_in,
                                               target_seq_len=2 * window_frames,
                                               dropout=0.3, out_dim=hidden_dim)
            self.sync_ca_m2a = nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout,
                                                     batch_first=True)
            self.sync_ca_a2m = nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout,
                                                     batch_first=True)
            self.sync_ln_m = nn.LayerNorm(hidden_dim)
            self.sync_ln_a = nn.LayerNorm(hidden_dim)
            self.sync_head = _mlp_head(4 * hidden_dim, 128, dropout)

        # ---- Fusion branch: the A9 experiment ------------------------
        # Both arms produce a 4*hidden vector and pass it through an
        # identically shaped MLP. The ONLY difference is cross-attention.
        if self.use_fusion_head:
            if use_cross_attention and not late_fusion_baseline:
                self.fuse_ca_v2a = nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout,
                                                         batch_first=True)
                self.fuse_ca_a2v = nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout,
                                                         batch_first=True)
                self.fuse_ln_v = nn.LayerNorm(hidden_dim)
                self.fuse_ln_a = nn.LayerNorm(hidden_dim)
            self.fusion_head = _mlp_head(4 * hidden_dim, 128, dropout)

    # ---- Backbone freezing ------------------------------------------
    def set_backbone_trainable(self, freeze_blocks):
        """features[:freeze_blocks] frozen (and their BatchNorms pinned to
        eval so the running stats of a 1,500-clip dataset cannot drift the
        pretrained low-level filters)."""
        self.freeze_blocks = freeze_blocks
        for i, block in enumerate(self.backbone):
            requires_grad = i >= freeze_blocks
            for p in block.parameters():
                p.requires_grad = requires_grad

    def train(self, mode=True):
        super().train(mode)
        if mode:
            for i, block in enumerate(self.backbone):
                if i < self.freeze_blocks:
                    block.eval()
        return self

    def backbone_param_groups(self, lr_backbone, llrd=0.75):
        """Layer-wise LR decay: later (more task-specific) blocks get the
        full backbone LR, earlier blocks get progressively less. Standard
        practice when fine-tuning a pretrained CNN on ~1k samples, and the
        main guard against the backbone collapsing onto identity cues."""
        groups, n = [], len(self.backbone)
        for i in range(self.freeze_blocks, n):
            params = [p for p in self.backbone[i].parameters() if p.requires_grad]
            if params:
                groups.append({"params": params, "lr": lr_backbone * (llrd ** (n - 1 - i))})
        return groups

    # ---- Visual encoding, split out so MC-Dropout can reuse it -------
    def encode_visual(self, crops):
        """crops: (B, T, S, S, 3) uint8 -> (V_tok, M_tok), each (B, T, 1280).

        V_tok is the grid mean (the global token, i.e. v3.3's input but now
        learned); M_tok is the BOTTOM ROW of the 3x3 grid, which is where
        the mouth sits in a margin-expanded face crop. Deterministic: the
        backbone contains no nn.Dropout, so MC-Dropout never needs to
        re-run it."""
        B, T = crops.shape[0], crops.shape[1]
        x = crops.reshape(B * T, *crops.shape[2:])            # (B*T, S, S, 3) uint8
        x = x.permute(0, 3, 1, 2).float().div_(255.0)         # (B*T, 3, S, S)
        x = (x - self.img_mean) / self.img_std
        fmap = self.backbone(x)                               # (B*T, 1280, h, w)
        grid = self.spatial_pool(fmap)                        # (B*T, 1280, 3, 3)
        v_tok = grid.mean(dim=(2, 3)).view(B, T, self.backbone_dim)
        m_tok = grid[:, :, 2, :].mean(dim=2).view(B, T, self.backbone_dim)
        return v_tok, m_tok

    def forward_from_tokens(self, v_tok, m_tok, mfcc, sync_audio=None, visual_mask=None):
        B, T = v_tok.shape[0], v_tok.shape[1]
        if visual_mask is None:
            visual_mask = torch.ones(B, T, dtype=torch.bool, device=v_tok.device)
        key_padding_mask = ~visual_mask       # True = PAD, what nn.Transformer expects

        # Visual branch -> video-only representation
        V = self.visual_proj(v_tok)
        V_enc = self.visual_encoder(V, src_key_padding_mask=key_padding_mask)
        v_rep = masked_mean_std(V_enc, visual_mask)
        video_logit = self.video_head(v_rep).squeeze(-1)

        # Audio branch -> audio-only representation (fixed length after the
        # TCN's AdaptiveAvgPool1d, so no mask is needed)
        A_enc = self.audio_encoder(mfcc)
        a_rep = masked_mean_std(A_enc, None)
        audio_logit = self.audio_head(a_rep).squeeze(-1)

        sync_logit, fusion_logit, attn_v2a, attn_a2v = None, None, None, None

        if self.use_sync_head and sync_audio is not None:
            M = self.mouth_proj(m_tok)
            S_enc = self.mel_encoder(sync_audio)
            ca_m, _ = self.sync_ca_m2a(query=M, key=S_enc, value=S_enc)
            fused_m = self.sync_ln_m(M + ca_m)
            ca_a, _ = self.sync_ca_a2m(query=S_enc, key=M, value=M,
                                       key_padding_mask=key_padding_mask)
            fused_s = self.sync_ln_a(S_enc + ca_a)
            sync_logit = self.sync_head(torch.cat(
                [masked_mean_std(fused_m, visual_mask), masked_mean_std(fused_s, None)],
                dim=-1)).squeeze(-1)

        if self.use_fusion_head:
            if self.use_cross_attention and not self.late_fusion_baseline:
                ca_v2a, attn_v2a = self.fuse_ca_v2a(query=V_enc, key=A_enc, value=A_enc)
                fused_v = self.fuse_ln_v(V_enc + ca_v2a)
                ca_a2v, attn_a2v = self.fuse_ca_a2v(query=A_enc, key=V_enc, value=V_enc,
                                                    key_padding_mask=key_padding_mask)
                fused_a = self.fuse_ln_a(A_enc + ca_a2v)
                fused = torch.cat([masked_mean_std(fused_v, visual_mask),
                                   masked_mean_std(fused_a, None)], dim=-1)
            else:
                # A9 baseline: same 4*hidden width, same head, no cross-attention.
                fused = torch.cat([v_rep, a_rep], dim=-1)
            fusion_logit = self.fusion_head(fused).squeeze(-1)

        return {"video_logit": video_logit, "audio_logit": audio_logit,
                "sync_logit": sync_logit, "fusion_logit": fusion_logit,
                "attn_v2a": attn_v2a, "attn_a2v": attn_a2v}

    def forward(self, crops, mfcc, sync_audio=None, visual_mask=None):
        v_tok, m_tok = self.encode_visual(crops)
        return self.forward_from_tokens(v_tok, m_tok, mfcc, sync_audio, visual_mask)


# ---- MC-Dropout uncertainty utilities ----------------------------------

def unwrap(model):
    """DataParallel-safe access to the underlying module."""
    return model.module if isinstance(model, nn.DataParallel) else model


def enable_mc_dropout(model):
    """Switch ONLY Dropout layers back on; BatchNorm stays in eval mode with
    frozen running stats. Call model.eval() first. Retained from v3.3."""
    for module in model.modules():
        if isinstance(module, nn.Dropout):
            module.train()


def mc_logits(model, crops, mfcc, sync_audio=None, visual_mask=None, n_samples=20, tokens=None):
    """N stochastic head passes over ONE deterministic backbone pass.

    Returns {'video': (n, B), 'audio': (n, B), 'sync': (n, B)?,
    'fusion': (n, B)?} of RAW logits. Callers do their own temperature
    scaling and sigmoid -- Cell 5 needs the logits, not probabilities, to
    fit temperature.

    Only the heads contain dropout, so re-running the backbone per sample
    would be 20x the compute for identical visual tokens.

    `tokens`: optional precomputed (v_tok, m_tok) from encode_visual. Pass
    this when scoring the SAME crops against MULTIPLE audio inputs -- e.g.
    the sync head's matched vs. mismatched pair in Cell 5 -- so the
    backbone runs once across both calls instead of once per call."""
    m = unwrap(model)
    m.eval()
    enable_mc_dropout(m)
    out = {}
    with torch.no_grad():
        v_tok, m_tok = m.encode_visual(crops) if tokens is None else tokens
        for _ in range(n_samples):
            o = m.forward_from_tokens(v_tok, m_tok, mfcc, sync_audio, visual_mask)
            for k_out, k_in in (("video", "video_logit"), ("audio", "audio_logit"),
                                ("sync", "sync_logit"), ("fusion", "fusion_logit")):
                if o[k_in] is not None:
                    out.setdefault(k_out, []).append(o[k_in].float().cpu().numpy())
    m.eval()                                           # restore dropout-off
    return {k: np.stack(v, axis=0) for k, v in out.items()}


def build_model(cfg, pretrained=True):
    """Single construction path shared by Cells 4/5/6/7/8, so an ablation
    checkpoint always loads into the architecture it was trained as."""
    return CrossFuseModelV4(
        window_frames=cfg["WINDOW_FRAMES"], sync_mel_frames=cfg["SYNC_MEL_FRAMES"],
        n_mels=cfg["N_MELS"], use_cross_attention=cfg["USE_CROSS_ATTENTION"],
        use_sync_head=cfg["USE_SYNC_HEAD"], use_fusion_head=cfg["USE_FUSION_HEAD"],
        late_fusion_baseline=cfg["LATE_FUSION_BASELINE"],
        sync_dense_aligned=cfg["SYNC_DENSE_ALIGNED"],
        freeze_blocks=cfg["FREEZE_BLOCKS"], pretrained=pretrained)


_probe = build_model(CONFIG)
_n_total = sum(p.numel() for p in _probe.parameters())
_n_train = sum(p.numel() for p in _probe.parameters() if p.requires_grad)
print(f"CrossFuse v4 architecture ready (backbone={CONFIG['BACKBONE']}, "
      f"CA={CONFIG['USE_CROSS_ATTENTION']}, sync={CONFIG['USE_SYNC_HEAD']}, "
      f"fusion={CONFIG['USE_FUSION_HEAD']}, late_fusion_baseline={CONFIG['LATE_FUSION_BASELINE']})")
print(f"  Parameters: {_n_total:,} total | {_n_train:,} trainable "
      f"(features[:{CONFIG['FREEZE_BLOCKS']}] frozen)")
del _probe

### Cell 3 (Stage A) — Face-Crop + Waveform Caching
Run this cell standalone, then Save Version and publish `/kaggle/working/crops_v4` as a Kaggle
Dataset before Stage B. One job per clip (~1,500 total) — no self-blend/compression replicas baked in
here; both move to train time in Cell 4. Retained from v3.3: collision-proof keys + a pre-extraction
uniqueness assert (the fix that recovered v3.2's ~290 silently-dropped clips), global union-find
identity resolution over the full pool before any sampling, the RVFA/FVRA/FVFA category caps, a
manifest-integrity assert against what's actually on disk, and 2-GPU sharding.

In [ ]:
# =====================================================================
# CELL 3 (STAGE A): FACE-CROP + WAVEFORM CACHING
# Run this cell standalone, then Save Version and publish
# /kaggle/working/crops_v4 as a Kaggle Dataset before Stage B (Cell 4+).
#
# WHAT CHANGED vs v3.3:
#
#  * WE CACHE PIXELS, NOT EMBEDDINGS. v3.3 wrote a frozen 1792-d
#    EfficientNet-B4 vector per frame, which permanently discarded
#    everything the model would later need to learn. We write raw uint8
#    160x160 crops plus the clip's 16 kHz waveform, so Cell 4 can fine-tune
#    a backbone and re-augment from scratch every epoch.
#
#  * ONE JOB PER CLIP. v3.3 expanded each clip into base + self-blend +
#    2 compression replicas, so ~1,500 independent clips became 5,000
#    "samples". That inflated every bootstrap CI (~1.7x too narrow, because
#    ~3x the independent observations were assumed) and let near-duplicates
#    cross split boundaries in the random-split ablation A8, confounding
#    identity leakage with replica leakage. Both augmentations move to
#    TRAIN TIME in Cell 4, where they are also strictly stronger: a fresh
#    random draw every epoch instead of two frozen copies.
#
#  * CONTIGUOUS FRAMES + TIMESTAMPS. Each clip stores an unbroken 64-frame
#    run and the exact wall-clock timestamp of every frame, so Cell 4 can
#    cut the audio window the frames actually cover. v3.3 strided across
#    the whole clip and had no shared time base with its audio at all.
#
#  * n_valid_faces now counts frames where MTCNN ACTUALLY detected a face
#    (frames that inherited a neighbour's box, or fell back to a centre
#    crop, are excluded), so Cell 4's attention mask reflects detection
#    quality rather than just clip length.
#
# RETAINED VERBATIM from v3.3 (these were correct and are load-bearing):
#  * Collision-proof keys f"{cat}_{identity}_{stem}_{md5(path)[:8]}" plus a
#    uniqueness assert BEFORE extraction -- this is the fix that recovered
#    the ~290 clips v3.2 silently dropped to basename collisions.
#  * Global union-find identity resolution over the FULL discovered pool
#    (Cell 1) BEFORE any sampling, so source-identity cross-references are
#    caught.
#  * Rebalanced category-stratified sampling: keep ALL RealVideo-FakeAudio,
#    cap FakeVideo-* at 250 each.
#  * Manifest-integrity assert against what is actually on disk.
#  * 2-GPU sharding via ThreadPoolExecutor.
# =====================================================================

import os
import csv
import glob
import numpy as np
import torch
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

CROP_DIR = "/kaggle/working/crops_v4"
os.makedirs(CROP_DIR, exist_ok=True)
manifest_path = os.path.join(CROP_DIR, "manifest.csv")


def get_fakeavceleb_label(video_path):
    path_norm = video_path.replace("\\", "/").lower()
    if "realvideo-realaudio" in path_norm:
        return 0
    elif any(f in path_norm for f in ["realvideo-fakeaudio", "fakevideo-realaudio",
                                      "fakevideo-fakeaudio"]):
        return 1
    elif "real" in os.path.basename(path_norm) and "fake" not in os.path.basename(path_norm):
        return 0
    return 1


def get_fakeavceleb_category(video_path):
    path_norm = video_path.replace("\\", "/")
    for cat in ["RealVideo-RealAudio", "RealVideo-FakeAudio",
                "FakeVideo-RealAudio", "FakeVideo-FakeAudio"]:
        if cat.lower() in path_norm.lower():
            return cat
    return "Unknown"


CATEGORY_SHORT = {
    "RealVideo-RealAudio": "RVRA", "RealVideo-FakeAudio": "RVFA",
    "FakeVideo-RealAudio": "FVRA", "FakeVideo-FakeAudio": "FVFA", "Unknown": "UNK",
}


def category_to_modality_labels(category):
    """(video_label, audio_label) for a category string. The 2x2 structure
    of FakeAVCeleb is what makes per-modality attribution possible at all."""
    c = category.lower()
    return (1 if "fakevideo" in c else 0), (1 if "fakeaudio" in c else 0)


def make_sample_key(category, identity_group, video_path):
    stem = os.path.basename(video_path).split('.')[0]
    rel_hash = hashlib.md5(video_path.replace("\\", "/").encode()).hexdigest()[:8]
    return f"{CATEGORY_SHORT.get(category, 'UNK')}_{identity_group}_{stem}_{rel_hash}"


# ---- 1. Locate the FakeAVCeleb pool ------------------------------------
all_videos = sorted(glob.glob("/kaggle/input/**/*.mp4", recursive=True))
if len(all_videos) == 0:
    all_videos = sorted(glob.glob("/kaggle/input/**/*.avi", recursive=True))

fav_videos = [v for v in all_videos
              if not any(t in v.lower() for t in ("dfdc", "deepfake-detection",
                                                  "celeb-df", "celebdf", "asvspoof"))]

real_videos, fake_videos = [], []
for v in fav_videos:
    (real_videos if get_fakeavceleb_label(v) == 0 else fake_videos).append(v)
print(f"Raw candidate pool: {len(real_videos)} real | {len(fake_videos)} fake")

# ---- 2. Global identity resolution BEFORE any sampling -----------------
print("Resolving identity groups (union-find over id-tokens in path + filename)...")
identity_groups = build_identity_groups(fav_videos)
print(f"  {len(fav_videos)} clips -> {len(set(identity_groups.values()))} identity groups")

rng_cat = make_rng("category_sample")
rng_cat.shuffle(real_videos)
rng_cat.shuffle(fake_videos)

selected_reals = real_videos          # the real pool is small; use all of it

fakes_by_cat = defaultdict(list)
for v in fake_videos:
    fakes_by_cat[get_fakeavceleb_category(v)].append(v)
for cat in fakes_by_cat:
    rng_cat.shuffle(fakes_by_cat[cat])

selected_fakes = []
selected_fakes += fakes_by_cat.get("RealVideo-FakeAudio", [])                     # keep ALL
selected_fakes += fakes_by_cat.get("FakeVideo-RealAudio", [])[:CONFIG["CAP_FVRA"]]
selected_fakes += fakes_by_cat.get("FakeVideo-FakeAudio", [])[:CONFIG["CAP_FVFA"]]
rng_cat.shuffle(selected_fakes)

if CONFIG["HELD_OUT_CATEGORY"] is not None:
    held = [v for v in selected_fakes if get_fakeavceleb_category(v) == CONFIG["HELD_OUT_CATEGORY"]]
    selected_fakes = [v for v in selected_fakes
                      if get_fakeavceleb_category(v) != CONFIG["HELD_OUT_CATEGORY"]]
    print(f"Holding out {len(held)} '{CONFIG['HELD_OUT_CATEGORY']}' videos entirely")

clip_list = selected_reals + selected_fakes
plan_counts = defaultdict(int)
for v in clip_list:
    plan_counts[get_fakeavceleb_category(v)] += 1
print(f"\nBase clips to cache: {len(clip_list)}  (v3.3 would have expanded these into "
      f"~{3 * len(clip_list) + len(selected_reals)} pseudo-samples)")
print(f"  per-category -> {dict(plan_counts)}")
est_gb = len(clip_list) * CONFIG["CACHED_FRAMES"] * (CONFIG["CROP_SIZE"] ** 2) * 3 / 1e9
print(f"  estimated crop storage: {est_gb:.2f} GB  (Kaggle notebook-output limit is 20 GB)")

print("\nSample identity parsing (these must look like real identity groups):")
for v in (selected_reals[:2] + selected_fakes[:3]):
    vl, al = category_to_modality_labels(get_fakeavceleb_category(v))
    print(f"  {os.path.basename(v)} -> identity_group='{identity_groups[v]}', "
          f"category='{get_fakeavceleb_category(v)}', video_label={vl}, audio_label={al}")

# ---- 3. Build jobs and assert key uniqueness BEFORE extracting ---------
jobs = []
for v in clip_list:
    cat = get_fakeavceleb_category(v)
    jobs.append(dict(video_path=v, category=cat, identity_group=identity_groups[v],
                     key=make_sample_key(cat, identity_groups[v], v)))

_keys = [j["key"] for j in jobs]
assert len(_keys) == len(set(_keys)), (
    f"Key collision in the extraction plan: {len(_keys) - len(set(_keys))} duplicates. "
    "This assert is the guard that recovered v3.2's ~290 silently-dropped clips.")
print(f"\nExtraction jobs queued: {len(jobs)} (all keys unique)")


def process_job(job, device_str, mtcnn_local):
    key = job["key"]
    crop_path = os.path.join(CROP_DIR, f"{key}.npy")
    audio_path = os.path.join(CROP_DIR, f"{key}_audio.npy")
    meta_path = os.path.join(CROP_DIR, f"{key}_meta.npz")
    if os.path.exists(crop_path) and os.path.exists(audio_path) and os.path.exists(meta_path):
        # Already cached from an earlier (possibly interrupted) run of this
        # cell -- reconstruct the manifest metadata from what's on disk
        # instead of just skipping it. Returning None here would silently
        # drop this clip from manifest_rows on every re-run, which is
        # EXACTLY the failure mode (v3.2's ~290 silently-dropped clips) the
        # uniqueness assert above and the integrity assert below exist to
        # catch -- a cache hit must not be a silent hole in the manifest.
        meta_npz = np.load(meta_path)
        valid_c, ts_c = meta_npz["valid"], meta_npz["timestamps"]
        y_c = np.load(audio_path)
        fps_c = float(1.0 / (ts_c[1] - ts_c[0])) if len(ts_c) > 1 and ts_c[1] > ts_c[0] else 25.0
        return job, "cached", dict(n_valid=int(valid_c.sum()), fps=fps_c, start_frame=-1,
                                   t0=float(ts_c[0]), t1=float(ts_c[-1]),
                                   duration=float(len(y_c) / CONFIG["AUDIO_SR"]))

    rng = make_rng(key)
    try:
        crops, valid, fps, start_frame, timestamps = extract_face_crops_contiguous(
            job["video_path"], mtcnn_local, rng=rng)
    except Exception as e:
        return job, f"visual_error:{type(e).__name__}", None
    if crops is None:
        return job, "no_frames", None
    n_valid = int(valid.sum())
    if n_valid == 0:
        return job, "no_faces", None

    try:
        y, sr = ffmpeg_extract_wav(job["video_path"])
    except Exception as e:
        return job, f"audio_error:{type(e).__name__}", None
    if y is None or len(y) < sr // 10:
        return job, "audio_too_short", None

    np.save(crop_path, crops)
    np.save(audio_path, np.asarray(y, dtype=np.float16))
    # valid + timestamps are PER FRAME (Cell 4's attention mask and audio
    # window-cutting both need per-frame data, not just a scalar count).
    np.savez(meta_path, valid=valid, timestamps=timestamps)
    return job, "ok", dict(n_valid=n_valid, fps=float(fps), start_frame=int(start_frame),
                           t0=float(timestamps[0]), t1=float(timestamps[-1]),
                           duration=float(len(y) / sr))


detectors = {d: build_detector(d) for d in DEVICE_IDS}
results_log = defaultdict(int)
manifest_rows = []

with ThreadPoolExecutor(max_workers=len(DEVICE_IDS)) as pool:
    futures = []
    for i, job in enumerate(jobs):
        device_str = DEVICE_IDS[i % len(DEVICE_IDS)]
        futures.append(pool.submit(process_job, job, device_str, detectors[device_str]))
    for fut in tqdm(as_completed(futures), total=len(futures), desc="Caching crops (v4, 2-GPU)"):
        job, status, meta = fut.result()
        results_log[status] += 1
        if status in ("ok", "cached"):
            v_lab, a_lab = category_to_modality_labels(job["category"])
            manifest_rows.append(dict(
                key=job["key"], path=job["video_path"], category=job["category"],
                identity_group=job["identity_group"], video_label=v_lab, audio_label=a_lab,
                n_valid_faces=meta["n_valid"], fps=round(meta["fps"], 3),
                start_frame=meta["start_frame"], t0=round(meta["t0"], 4),
                t1=round(meta["t1"], 4), duration=round(meta["duration"], 3)))

print(f"\nExtraction outcomes: {dict(results_log)}")

# ---- 4. Manifest integrity check ---------------------------------------
with open(manifest_path, "w", newline="") as f:
    fieldnames = ["key", "path", "category", "identity_group", "video_label", "audio_label",
                  "n_valid_faces", "fps", "start_frame", "t0", "t1", "duration"]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(manifest_rows)

on_disk = {os.path.splitext(os.path.basename(p))[0]
           for p in glob.glob(os.path.join(CROP_DIR, "*.npy"))
           if not p.endswith("_audio.npy")}
manifest_keys = {r["key"] for r in manifest_rows}
assert len(manifest_rows) == len(manifest_keys), "Duplicate keys in manifest!"
missing = manifest_keys - on_disk
assert not missing, f"{len(missing)} manifest rows have no .npy on disk: {list(missing)[:5]}"
for r in manifest_rows:
    assert os.path.exists(os.path.join(CROP_DIR, r["key"] + "_audio.npy")), \
        f"Missing waveform for {r['key']}"
    assert os.path.exists(os.path.join(CROP_DIR, r["key"] + "_meta.npz")), \
        f"Missing meta (valid/timestamps) for {r['key']}"

total_bytes = sum(os.path.getsize(p) for p in glob.glob(os.path.join(CROP_DIR, "*.npy")))
cat_final = defaultdict(int)
vlab_final = defaultdict(int)
alab_final = defaultdict(int)
for r in manifest_rows:
    cat_final[r["category"]] += 1
    vlab_final[r["video_label"]] += 1
    alab_final[r["audio_label"]] += 1

print(f"Manifest integrity OK -> {len(manifest_rows)} rows, {len(manifest_keys)} unique keys, "
      f"{len(on_disk)} crop files on disk.")
print(f"Per-category counts: {dict(cat_final)}")
print(f"Video real/fake: {vlab_final[0]}/{vlab_final[1]} | "
      f"Audio real/fake: {alab_final[0]}/{alab_final[1]}")
print(f"Total cache size: {total_bytes / 1e9:.2f} GB")
if total_bytes / 1e9 > 19.0:
    print("[!] Approaching Kaggle's 20 GB notebook-output limit -- lower CACHED_FRAMES "
          "or CROP_SIZE and re-run.")

# ---- 5. Visual spot-check (verification gate from the plan) ------------
if manifest_rows:
    fig, axes = plt.subplots(1, 5, figsize=(15, 3.2))
    for ax, r in zip(axes, manifest_rows[:5]):
        arr = np.load(os.path.join(CROP_DIR, r["key"] + ".npy"), mmap_mode="r")
        assert arr.shape == (CONFIG["CACHED_FRAMES"], CONFIG["CROP_SIZE"],
                             CONFIG["CROP_SIZE"], 3), f"Bad shape {arr.shape} for {r['key']}"
        assert arr.dtype == np.uint8, f"Bad dtype {arr.dtype} for {r['key']}"
        ax.imshow(arr[0]); ax.axis("off")
        ax.set_title(f"{r['category'][:12]}\nvalid {r['n_valid_faces']}/{CONFIG['CACHED_FRAMES']}",
                     fontsize=8)
    plt.suptitle("Cached crop spot-check -- these must be recognisable faces", fontsize=11)
    plt.tight_layout(); plt.show()

print(f"\nManifest written to: {manifest_path}")
print("Stage A complete. Save Version, then publish /kaggle/working/crops_v4 as a Kaggle "
      "Dataset and attach it before running Stage B (Cell 4 onward).")

### Cell 4 (Stage B start) — Sample-Proportional Split, Train-Time Augmentation,
### Two-Phase Fine-Tuning
Attach the `crops_v4` Kaggle Dataset published by Cell 3 first. `build_3way_split_packed` targets
60/20/20 by SAMPLE count (not group count, unlike v3.3's `GroupShuffleSplit`), so every ablation row
in Cell 8 is scored on comparably-sized splits. Training is two-phase — heads only over a frozen
backbone, then a full fine-tune with layer-wise LR decay — with self-blend/compression augmentation
generated fresh per epoch, train-split only. Sync negatives are identity-aware (never a blind
`torch.roll` on an unshuffled loader, which mislabelled true matches as "mismatched" in v3.3).
`run_training_pipeline()` now reloads the BEST checkpoint before returning — the single fix that
invalidates v3.3's entire ablation table (every row there was scored ~7 epochs past its own best
checkpoint). Runs the main config across `N_SEEDS` seeds, reporting mean ± std.

In [ ]:
# =====================================================================
# CELL 4 (STAGE B START): SAMPLE-PROPORTIONAL IDENTITY-DISJOINT SPLIT,
#         TRAIN-TIME AUGMENTATION, AND TWO-PHASE FINE-TUNING
# Attach the crops_v4 Kaggle Dataset published by Cell 3 before running.
#
# WHAT CHANGED vs v3.3, and why:
#
#  1. THE BACKBONE IS TRAINED. Phase 1 trains the heads over a frozen
#     backbone (fast, stabilises the randomly-initialised heads so their
#     gradients cannot wreck pretrained filters); Phase 2 unfreezes
#     features[FREEZE_BLOCKS:] with LAYER-WISE LR DECAY. This is the fix
#     for the finding that eight different v3.3 architectures all scored
#     within 0.03 AUC of each other -- the representation was frozen, so
#     nothing above it could matter.
#
#  2. AUGMENTATION IS FRESH EVERY EPOCH, NOT TWO BAKED REPLICAS.
#     Because Cell 3 now caches pixels, compression/quality augmentation
#     and self-blend hard negatives are generated on the fly. Beyond being
#     a much stronger regulariser, this removes the pseudo-replication that
#     made every v3.3 bootstrap CI ~1.7x too narrow.
#     SELF-BLEND IS TRAIN-SPLIT-ONLY -- val and test see only real footage.
#     (v3.3 wrote self-blends into the cache, so they landed in val/test
#     too and were scored as if they were genuine FakeAVCeleb fakes.)
#
#  3. SPLITS ARE PROPORTIONAL BY SAMPLE COUNT, NOT GROUP COUNT.
#     v3.3's GroupShuffleSplit(train_size=0.85) splits by the number of
#     GROUPS, so the main run got 946 test samples and ablation A8 got 750 --
#     rows measured on different-sized sets are not comparable. The greedy
#     packer below hits the target SAMPLE fractions and balances category
#     composition at the same time.
#
#  4. SYNC NEGATIVES ARE IDENTITY-AWARE AND INCLUDE TEMPORAL OFFSETS.
#     v3.3 built negatives with torch.roll(shifts=1) on an UNSHUFFLED
#     val/test loader, so a clip was frequently paired with its own
#     augmented replica -- identical audio, labelled "mismatched". Here
#     negatives are (a) audio from a DIFFERENT identity group, and (b)
#     SyncNet-style audio from the SAME clip shifted >= 0.4s, which is the
#     negative that actually forces lip-sync rather than speaker identity.
#
#  5. run_training_pipeline() RELOADS THE BEST CHECKPOINT BEFORE RETURNING.
#     v3.3 saved the best state to disk but returned the model in its
#     FINAL-epoch state, and Cell 8 evaluated that -- every ablation row was
#     scored ~7 epochs past its best checkpoint. That single bug is why
#     A2_audio_only reported 0.6265 for a head that reaches 0.997 elsewhere.
#
#  6. pos_weight ONLY, NO WeightedRandomSampler. v3.3 applied both, which
#     double-corrects the same imbalance. pos_weight is the clean choice: it
#     reweights the loss without distorting what an epoch contains.
#
# RETAINED from v3.3: the three-way pairwise identity-overlap assert, CMVN
# at load time (so A7 stays a toggle), masked training, label smoothing,
# cosine schedule with warmup, early stopping, and the single
# run_training_pipeline() entry point that Cell 8 reuses.
# =====================================================================

import os
import csv
import math
import json
import glob
import copy
import random
import numpy as np
import torch
import torch.nn as nn
import cv2
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader

CROP_DIR = "/kaggle/working/crops_v4"
if not os.path.isdir(CROP_DIR) or not glob.glob(os.path.join(CROP_DIR, "*_audio.npy")):
    candidates = sorted(glob.glob("/kaggle/input/*/crops_v4")) + \
                 sorted(glob.glob("/kaggle/input/*crops*v4*"))
    if candidates:
        CROP_DIR = candidates[0]
print(f"[i] CROP_DIR = {CROP_DIR}")
MANIFEST_PATH = os.path.join(CROP_DIR, "manifest.csv")

with open(MANIFEST_PATH, newline="") as f:
    MANIFEST = list(csv.DictReader(f))
for r in MANIFEST:
    r["video_label"] = int(r["video_label"])
    r["audio_label"] = int(r["audio_label"])
    r["n_valid_faces"] = int(r["n_valid_faces"])
    r["fps"] = float(r["fps"])
print(f"[i] Manifest: {len(MANIFEST)} clips | video real/fake "
      f"{sum(1 for r in MANIFEST if r['video_label'] == 0)}/"
      f"{sum(1 for r in MANIFEST if r['video_label'] == 1)} | audio real/fake "
      f"{sum(1 for r in MANIFEST if r['audio_label'] == 0)}/"
      f"{sum(1 for r in MANIFEST if r['audio_label'] == 1)}")

GROUP_TO_ID = {g: i for i, g in enumerate(sorted({r["identity_group"] for r in MANIFEST}))}


# ---------------------------------------------------------------------
# Split: greedy group packing to SAMPLE-count targets, category-balanced
# ---------------------------------------------------------------------
def build_3way_split_packed(rows, identity_disjoint=True, fracs=(0.60, 0.20, 0.20), seed=42):
    """Assign whole identity groups to train/val/test so that each split
    hits its target fraction of SAMPLES (not of groups) while keeping the
    4-way category mix close to the population mix.

    Greedy: process groups largest-first and put each into whichever split
    minimises a squared relative-deviation cost over per-category targets.
    Largest-first matters -- a big identity group placed late cannot be
    compensated for, which is exactly how GroupShuffleSplit ended up with
    946 vs 750 test samples across otherwise-identical configurations.

    identity_disjoint=False (ablation A8) makes every clip its own group.
    Note that with v4's one-row-per-clip cache this is now a clean
    manipulation: there are no augmented replicas left to leak across the
    boundary, so the A4-vs-A8 gap isolates IDENTITY leakage alone. In v3.3
    it confounded identity leakage with near-duplicate leakage.
    """
    n = len(rows)
    cats = sorted({r["category"] for r in rows})
    cat_idx = {c: i for i, c in enumerate(cats)}

    groups = defaultdict(list)
    for i, r in enumerate(rows):
        key = r["identity_group"] if identity_disjoint else f"__clip_{i}"
        groups[key].append(i)

    pop = np.zeros(len(cats))
    for r in rows:
        pop[cat_idx[r["category"]]] += 1
    targets = np.stack([np.asarray(f) * pop for f in fracs])          # (3, n_cats)
    targets = np.maximum(targets, 1e-6)

    rng = random.Random(seed)
    order = sorted(groups.keys(),
                   key=lambda g: (-len(groups[g]),
                                  hashlib.md5(f"{g}:{seed}".encode()).hexdigest()))
    cur = np.zeros((3, len(cats)))
    assign = {}
    for g in order:
        vec = np.zeros(len(cats))
        for i in groups[g]:
            vec[cat_idx[rows[i]["category"]]] += 1
        costs = []
        for s in range(3):
            trial = cur.copy()
            trial[s] += vec
            costs.append(float(np.sum(((trial - targets) / targets) ** 2)))
        best = int(np.argmin(costs))
        assign[g] = best
        cur[best] += vec

    split_idx = [[], [], []]
    for g, s in assign.items():
        split_idx[s].extend(groups[g])
    train_idx, val_idx, test_idx = (np.array(sorted(s), dtype=int) for s in split_idx)

    if identity_disjoint:
        ids = [set(rows[i]["identity_group"] for i in idx) for idx in (train_idx, val_idx, test_idx)]
        assert not (ids[0] & ids[1]) and not (ids[0] & ids[2]) and not (ids[1] & ids[2]), \
            "Identity leakage across the 3-way split -- check identity_group in Cell 3!"

    for name, idx in zip(("train", "val", "test"), (train_idx, val_idx, test_idx)):
        comp = defaultdict(int)
        for i in idx:
            comp[rows[i]["category"]] += 1
        print(f"    {name:<5} n={len(idx):>4} ({len(idx) / n:.1%})  {dict(comp)}")
    return train_idx, val_idx, test_idx


# ---------------------------------------------------------------------
# Train-time augmentation (moved out of Stage A -- see header note 2)
# ---------------------------------------------------------------------
def _soft_ellipse_mask(h, w, rng):
    """Feathered elliptical mask over the face region of an already-tight
    crop. v3.3 built this from MTCNN landmarks during extraction; on a
    margin-expanded, face-centred 160x160 crop an ellipse covers the same
    region without needing a landmark pass per training sample, and the
    thing self-blend is meant to teach -- a soft blending BOUNDARY -- is
    produced just as well."""
    mask = np.zeros((h, w), dtype=np.uint8)
    cx = int(w * rng.uniform(0.45, 0.55))
    cy = int(h * rng.uniform(0.45, 0.58))
    ax = int(w * rng.uniform(0.28, 0.40))
    ay = int(h * rng.uniform(0.34, 0.46))
    cv2.ellipse(mask, (cx, cy), (ax, ay), rng.uniform(-15, 15), 0, 360, 255, -1)
    k = 2 * rng.randint(6, 14) + 1
    return (cv2.GaussianBlur(mask, (k, k), 0).astype(np.float32) / 255.0)[..., None]


def self_blend_clip(window, rng):
    """Blend each frame with a colour/affine-jittered copy of itself through
    a soft elliptical mask -> a synthetic VIDEO-fake carrying a realistic
    blending boundary but no generator fingerprint. Warp parameters are
    drawn ONCE per clip so the artefact is temporally coherent (a per-frame
    redraw would create a flicker the model could trivially latch onto)."""
    T, h, w, _ = window.shape
    mask = _soft_ellipse_mask(h, w, rng)
    angle = rng.uniform(-4, 4)
    tx, ty = rng.uniform(-0.02, 0.02) * w, rng.uniform(-0.02, 0.02) * h
    scale = rng.uniform(0.97, 1.03)
    gain = rng.uniform(0.90, 1.10)
    bias = rng.uniform(-12, 12)
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, scale)
    M[0, 2] += tx
    M[1, 2] += ty
    out = np.empty_like(window)
    for t in range(T):
        base = window[t].astype(np.float32)
        src = cv2.warpAffine(window[t], M, (w, h), flags=cv2.INTER_LINEAR,
                             borderMode=cv2.BORDER_REFLECT).astype(np.float32)
        src = np.clip(src * gain + bias, 0, 255)
        out[t] = np.clip(mask * src + (1.0 - mask) * base, 0, 255).astype(np.uint8)
    return out


def compression_augment_clip(window, rng):
    """Random downscale->upscale + JPEG re-encode + mild noise, with ONE
    parameter draw shared across the clip (a real re-encode is
    clip-consistent). This is what gives the video head examples of the
    compression artefacts in-the-wild footage actually has; v3.3 baked
    exactly two such copies into the cache, whereas this redraws every
    epoch."""
    T, h, w, _ = window.shape
    out = window
    if rng.random() < 0.8:
        s = rng.uniform(0.4, 1.0)
        nh, nw = max(16, int(h * s)), max(16, int(w * s))
        out = np.stack([cv2.resize(cv2.resize(f, (nw, nh), interpolation=cv2.INTER_AREA),
                                   (w, h), interpolation=cv2.INTER_LINEAR) for f in out])
    if rng.random() < 0.8:
        q = rng.randint(30, 90)
        enc_param = [int(cv2.IMWRITE_JPEG_QUALITY), q]
        frames = []
        for f in out:
            ok, buf = cv2.imencode(".jpg", f[:, :, ::-1], enc_param)
            frames.append(cv2.imdecode(buf, cv2.IMREAD_COLOR)[:, :, ::-1] if ok else f)
        out = np.stack(frames)
    if rng.random() < 0.5:
        sigma = rng.uniform(1.0, 5.0)
        noise = np.random.RandomState(rng.randint(0, 2**31 - 1)).normal(0, sigma, out.shape)
        out = np.clip(out.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    return out


def photometric_augment_clip(window, rng):
    """Clip-consistent brightness/contrast jitter + horizontal flip + a
    single random-erasing box."""
    out = window
    if rng.random() < 0.5:
        out = out[:, :, ::-1, :]
    if rng.random() < 0.8:
        gain, bias = rng.uniform(0.85, 1.15), rng.uniform(-18, 18)
        out = np.clip(out.astype(np.float32) * gain + bias, 0, 255).astype(np.uint8)
    if rng.random() < 0.3:
        T, h, w, _ = out.shape
        eh, ew = rng.randint(h // 8, h // 4), rng.randint(w // 8, w // 4)
        y0, x0 = rng.randint(0, h - eh), rng.randint(0, w - ew)
        out = out.copy()
        out[:, y0:y0 + eh, x0:x0 + ew, :] = rng.randint(0, 255)   # inclusive both ends -- 256 would overflow uint8
    return np.ascontiguousarray(out)


# ---------------------------------------------------------------------
# Dataset
# ---------------------------------------------------------------------
class CrossFuseCropDataset(Dataset):
    """Returns one contiguous WINDOW_FRAMES window of cached crops plus the
    audio that covers exactly the same seconds.

    Train mode draws a random window start and applies fresh augmentation;
    eval mode always takes the CENTRE window with no augmentation, so val
    and test are deterministic across epochs, seeds and ablation rows.
    """

    SELFBLEND_P = 0.25

    def __init__(self, rows, indices, crop_dir, cfg, training=False):
        self.rows = rows
        # int(x), not list(indices): indices arrives as an np.array(dtype=int),
        # and list() on that yields numpy.int64 scalars, not native Python
        # int. __getitem__ below seeds random.Random() from an expression
        # built on ridx = self.indices[i] -- random.seed() strictly requires
        # (None, int, float, str, bytes, bytearray), and numpy.int64 fails
        # that isinstance check with a TypeError raised inside the DataLoader
        # worker on the very first batch.
        self.indices = [int(x) for x in indices]
        self.crop_dir = crop_dir
        self.cfg = cfg
        self.training = training
        self.W = cfg["WINDOW_FRAMES"]
        self.sr = cfg["AUDIO_SR"]

    def __len__(self):
        return len(self.indices)

    def _load(self, ridx):
        r = self.rows[ridx]
        crops = np.load(os.path.join(self.crop_dir, r["key"] + ".npy"), mmap_mode="r")
        y = np.load(os.path.join(self.crop_dir, r["key"] + "_audio.npy")).astype(np.float32)
        meta = np.load(os.path.join(self.crop_dir, r["key"] + "_meta.npz"))
        return r, crops, y, meta["valid"], meta["timestamps"]

    def __getitem__(self, i):
        ridx = self.indices[i]
        r, crops, y, valid, ts = self._load(ridx)
        rng = random.Random((self.cfg["SEED"] * 1_000_003 + ridx * 9176 + (i if self.training else 0))
                            ^ (random.getrandbits(30) if self.training else 0))

        T_cached = crops.shape[0]
        W = min(self.W, T_cached)
        max_start = max(0, T_cached - W)
        start = rng.randint(0, max_start) if self.training else max_start // 2
        window = np.array(crops[start:start + W])
        win_valid = np.array(valid[start:start + W], dtype=bool)
        fps = r["fps"] if r["fps"] > 1 else 25.0
        t0 = float(ts[start])
        t1 = t0 + W / fps

        v_label, a_label = r["video_label"], r["audio_label"]

        if self.training:
            # --- self-blend hard negative: a REAL visual track becomes fake
            if (self.cfg["GENERATE_HARD_NEGATIVES"] and v_label == 0
                    and rng.random() < self.SELFBLEND_P):
                window = self_blend_clip(window, rng)
                v_label = 1
            if self.cfg["USE_COMPRESSION_AUG"]:
                window = compression_augment_clip(window, rng)
            window = photometric_augment_clip(window, rng)
            # --- frame dropout: zero the frame AND mask it out
            drop = np.random.RandomState(rng.randint(0, 2**31 - 1)).rand(W) < 0.10
            drop &= win_valid
            if drop.any() and (win_valid & ~drop).any():
                window = window.copy()
                window[drop] = 0
                win_valid = win_valid & ~drop

        if not win_valid.any():
            win_valid[0] = True

        # --- audio: whole-clip MFCC for the forgery head, window-aligned
        #     log-mel for the sync head (see Cell 1's logmel_window).
        mfcc = maybe_cmvn(mfcc_from_waveform(y, self.sr), use_cmvn=self.cfg["USE_MFCC_CMVN"])
        if self.cfg["SYNC_DENSE_ALIGNED"]:
            sync_a = logmel_window(y, self.sr, t0, t1)
            # SyncNet-style hard negative: same clip, same speaker, same
            # recording conditions -- ONLY the time offset differs, so the
            # only way to tell it apart is actual lip-sync.
            dur = len(y) / self.sr
            shift = rng.choice([-1, 1]) * rng.uniform(0.4, max(0.5, dur / 3))
            o0 = float(np.clip(t0 + shift, 0.0, max(0.0, dur - (t1 - t0))))
            sync_off = logmel_window(y, self.sr, o0, o0 + (t1 - t0))
        else:
            # Ablation A10: the v3.3 arrangement -- whole-clip MFCC against a
            # visual window it has no time correspondence with.
            sync_a = mfcc.copy()
            sync_off = mfcc.copy()

        pad = self.W - W
        if pad > 0:
            window = np.concatenate([window, np.zeros((pad, *window.shape[1:]), np.uint8)], 0)
            win_valid = np.concatenate([win_valid, np.zeros(pad, bool)])

        return (torch.from_numpy(np.ascontiguousarray(window)),
                torch.from_numpy(mfcc),
                torch.from_numpy(sync_a),
                torch.from_numpy(sync_off),
                torch.tensor(float(v_label)), torch.tensor(float(a_label)),
                torch.from_numpy(win_valid),
                torch.tensor(GROUP_TO_ID[r["identity_group"]], dtype=torch.long))


def worker_init_fn(worker_id):
    base = CONFIG["SEED"] + worker_id
    random.seed(base)
    np.random.seed(base)
    torch.manual_seed(base)


def safe_auc(y_true, y_score):
    try:
        if len(set(int(v) for v in y_true)) < 2:
            return float('nan')
        return roc_auc_score(y_true, y_score)
    except ValueError:
        return float('nan')


def smooth_labels(y, eps=0.05):
    return y * (1.0 - eps) + 0.5 * eps


# ---------------------------------------------------------------------
# Training pipeline
# ---------------------------------------------------------------------
def run_training_pipeline(cfg, tag="main", seed=None, verbose=True):
    """Two-phase fine-tune. Returns a result dict whose "model" holds the
    BEST checkpoint's weights (reloaded before return -- see header note 5).
    Cell 8 calls this with modified CONFIG copies."""
    seed = cfg["SEED"] if seed is None else seed
    seed_everything(seed)
    cfg = dict(cfg, SEED=seed)

    if verbose:
        print(f"[{tag}] Split (identity_disjoint={cfg['IDENTITY_DISJOINT_SPLIT']}, "
              f"target {tuple(f'{f:.0%}' for f in cfg['SPLIT_FRACS'])} by SAMPLE count):")
    train_idx, val_idx, test_idx = build_3way_split_packed(
        MANIFEST, identity_disjoint=cfg["IDENTITY_DISJOINT_SPLIT"],
        fracs=cfg["SPLIT_FRACS"], seed=seed)

    train_ds = CrossFuseCropDataset(MANIFEST, train_idx, CROP_DIR, cfg, training=True)
    val_ds = CrossFuseCropDataset(MANIFEST, val_idx, CROP_DIR, cfg, training=False)
    test_ds = CrossFuseCropDataset(MANIFEST, test_idx, CROP_DIR, cfg, training=False)

    nw = cfg["NUM_WORKERS"]
    common = dict(num_workers=nw, pin_memory=True, worker_init_fn=worker_init_fn)
    train_loader = DataLoader(train_ds, batch_size=cfg["BATCH_CLIPS"], shuffle=True,
                              drop_last=True, persistent_workers=(nw > 0), **common)
    val_loader_l = DataLoader(val_ds, batch_size=cfg["BATCH_CLIPS"], shuffle=False, **common)
    test_loader_l = DataLoader(test_ds, batch_size=cfg["BATCH_CLIPS"], shuffle=False, **common)

    model_local = build_model(cfg).to(device)
    modality = cfg.get("MODALITY", "both")

    # pos_weight only (no sampler -- see header note 6). The expected count
    # accounts for self-blend converting a fraction of real videos to fakes.
    tv = [MANIFEST[i]["video_label"] for i in train_idx]
    n_pos, n_neg = sum(tv), len(tv) - sum(tv)
    if cfg["GENERATE_HARD_NEGATIVES"]:
        conv = CrossFuseCropDataset.SELFBLEND_P * n_neg
        n_pos, n_neg = n_pos + conv, n_neg - conv
    pos_weight = torch.tensor(max(n_neg, 1.0) / max(n_pos, 1.0), dtype=torch.float32).to(device)
    crit_video = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    crit_plain = nn.BCEWithLogitsLoss()
    LAMBDA_SYNC, LAMBDA_FUSION = 0.5, 0.5

    total_epochs = cfg["EPOCHS_FROZEN"] + cfg["EPOCHS_FINETUNE"]
    scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())
    ckpt_path = f"/kaggle/working/crossfuse_v4_{tag}_s{seed}.pth"

    def make_optim(phase):
        if phase == 0:
            model_local.set_backbone_trainable(len(model_local.backbone))   # everything frozen
            groups = [{"params": [p for n_, p in model_local.named_parameters()
                                  if p.requires_grad and not n_.startswith("backbone")],
                       "lr": cfg["LR_HEAD"] * 10}]
        else:
            model_local.set_backbone_trainable(cfg["FREEZE_BLOCKS"])
            groups = model_local.backbone_param_groups(cfg["LR_BACKBONE"], cfg["LLRD"])
            groups.append({"params": [p for n_, p in model_local.named_parameters()
                                      if p.requires_grad and not n_.startswith("backbone")],
                           "lr": cfg["LR_HEAD"]})
        return torch.optim.AdamW(groups, weight_decay=1e-4)

    optimizer = make_optim(0)
    best = {"mean_auc": -1.0, "epoch": -1, "video": float('nan'), "audio": float('nan'),
            "sync": float('nan'), "fusion": float('nan'), "train_video": float('nan'),
            "train_sync": float('nan')}
    since_best = 0

    if verbose:
        print(f"[{tag}] Training (modality={modality}, CA={cfg['USE_CROSS_ATTENTION']}, "
              f"sync={cfg['USE_SYNC_HEAD']}, fusion={cfg['USE_FUSION_HEAD']}, "
              f"late_fusion={cfg['LATE_FUSION_BASELINE']}, dense_sync={cfg['SYNC_DENSE_ALIGNED']}, "
              f"seed={seed}) -- phase 1 frozen x{cfg['EPOCHS_FROZEN']}, "
              f"phase 2 fine-tune x{cfg['EPOCHS_FINETUNE']}")

    for epoch in range(total_epochs):
        phase = 0 if epoch < cfg["EPOCHS_FROZEN"] else 1
        if epoch == cfg["EPOCHS_FROZEN"]:
            optimizer = make_optim(1)
            if verbose:
                n_tr = sum(p.numel() for p in model_local.parameters() if p.requires_grad)
                print(f"[{tag}] --- unfreezing backbone from block {cfg['FREEZE_BLOCKS']} "
                      f"({n_tr:,} trainable params) ---")
        if phase == 1:
            # Cosine decay with a 2-epoch warmup, scaled per-group off each
            # group's OWN base LR (set once, right after make_optim(1)
            # creates fresh groups) -- this is what keeps layer-wise LR
            # decay intact across the whole schedule instead of collapsing
            # every group to the same curve.
            prog = (epoch - cfg["EPOCHS_FROZEN"]) / max(1, cfg["EPOCHS_FINETUNE"])
            warm = min(1.0, (epoch - cfg["EPOCHS_FROZEN"] + 1) / 2.0)
            factor = max(warm * 0.5 * (1 + math.cos(math.pi * min(prog, 1.0))), 0.02)
            for g in optimizer.param_groups:
                g.setdefault("initial_lr", g["lr"])
                g["lr"] = g["initial_lr"] * factor

        model_local.train()
        run_loss, nb = 0.0, 0
        tr = defaultdict(list)
        optimizer.zero_grad(set_to_none=True)

        for step, batch in enumerate(train_loader):
            crops, mfcc, sync_a, sync_off, vlab, alab, mask, gid = [b.to(device, non_blocking=True)
                                                                    for b in batch]
            with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                # encode_visual ONCE, then run the heads off the shared tokens.
                # The sync loss needs a second head pass on the SAME crops with
                # MISMATCHED audio; calling model_local(crops, ...) a second
                # time would re-run the whole EfficientNet backbone on
                # identical pixels -- doubling the dominant cost of every step
                # for zero benefit. Gradients still flow into the backbone from
                # both head passes through the shared v_tok/m_tok.
                v_tok, m_tok = model_local.encode_visual(crops)
                out = model_local.forward_from_tokens(v_tok, m_tok, mfcc, sync_a, visual_mask=mask)
                loss = torch.zeros((), device=device)
                if modality in ("both", "video_only"):
                    loss = loss + crit_video(out["video_logit"], smooth_labels(vlab, 0.05))
                if modality in ("both", "audio_only"):
                    loss = loss + crit_plain(out["audio_logit"], alab)

                if out["fusion_logit"] is not None and modality == "both":
                    any_lab = torch.clamp(vlab + alab, 0, 1)
                    loss = loss + LAMBDA_FUSION * crit_plain(out["fusion_logit"], any_lab)

                if out["sync_logit"] is not None and modality == "both":
                    B = crops.size(0)
                    # Negative 1: audio from a DIFFERENT identity group.
                    perm = torch.randperm(B, device=device)
                    ok = gid[perm] != gid
                    # Negative 2: SAME clip, time-shifted >= 0.4s.
                    neg_a = torch.where(ok.view(-1, 1, 1), sync_a[perm], sync_off)
                    out_neg = model_local.forward_from_tokens(v_tok, m_tok, mfcc, neg_a,
                                                             visual_mask=mask)
                    s_logit = torch.cat([out["sync_logit"], out_neg["sync_logit"]], 0)
                    s_lab = torch.cat([torch.zeros(B, device=device),
                                       torch.ones(B, device=device)], 0)
                    loss = loss + LAMBDA_SYNC * crit_plain(s_logit, s_lab)
                    tr["s_prob"].extend(torch.sigmoid(s_logit.detach().float()).cpu().numpy())
                    tr["s_true"].extend([0] * B + [1] * B)

            scaler.scale(loss / cfg["GRAD_ACCUM"]).backward()
            if (step + 1) % cfg["GRAD_ACCUM"] == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model_local.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            run_loss += float(loss.detach()); nb += 1
            tr["v_prob"].extend(torch.sigmoid(out["video_logit"].detach().float()).cpu().numpy())
            tr["v_true"].extend(vlab.detach().cpu().numpy())
            tr["a_prob"].extend(torch.sigmoid(out["audio_logit"].detach().float()).cpu().numpy())
            tr["a_true"].extend(alab.detach().cpu().numpy())

        val = evaluate_deterministic(model_local, val_loader_l, cfg)
        train_video_auc = safe_auc(tr["v_true"], tr["v_prob"])
        train_sync_auc = safe_auc(tr["s_true"], tr["s_prob"]) if tr["s_true"] else float('nan')

        if modality == "video_only":
            mean_auc = val["video_auc"]
        elif modality == "audio_only":
            mean_auc = val["audio_auc"]
        else:
            # Selection deliberately uses ONLY the two attribution heads, so
            # the A9 fusion arms are checkpoint-selected identically and the
            # comparison between them stays fair.
            mean_auc = 0.5 * (val["video_auc"] + val["audio_auc"])

        improved = mean_auc > best["mean_auc"]
        if improved:
            best.update(mean_auc=mean_auc, epoch=epoch, video=val["video_auc"],
                        audio=val["audio_auc"], sync=val["sync_auc"], fusion=val["fusion_auc"],
                        train_video=train_video_auc, train_sync=train_sync_auc)
            since_best = 0
            torch.save(model_local.state_dict(), ckpt_path)
        else:
            since_best += 1

        if verbose:
            print(f"[{tag}] Ep [{epoch + 1:02d}/{total_epochs}] p{phase} "
                  f"lr {optimizer.param_groups[-1]['lr']:.2e} loss {run_loss / max(nb, 1):.4f} | "
                  f"train v/s {train_video_auc:.4f}/{train_sync_auc:.4f} | "
                  f"val v/a/s/f {val['video_auc']:.4f}/{val['audio_auc']:.4f}/"
                  f"{val['sync_auc']:.4f}/{val['fusion_auc']:.4f} | mean {mean_auc:.4f} | "
                  f"gap {train_video_auc - val['video_auc']:+.4f}{' * (New Best)' if improved else ''}")

        if since_best >= cfg["PATIENCE"] and phase == 1:
            if verbose:
                print(f"[{tag}] Early stopping at epoch {epoch + 1}.")
            break

    # ---- THE FIX: return the BEST weights, not the last epoch's ----------
    model_local.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=False))
    model_local.eval()

    if verbose:
        print(f"[{tag}] Done. Best epoch {best['epoch'] + 1} -> val video {best['video']:.4f} | "
              f"audio {best['audio']:.4f} | sync {best['sync']:.4f} | fusion {best['fusion']:.4f}")
        gap = best["train_video"] - best["video"]
        print(f"[{tag}] Video train/val gap: {gap:+.4f} "
              f"({'OK' if gap < 0.10 else 'HIGH -- check that train-time augmentation is firing'}; "
              f"v3.3 was +0.2473)")
        if not np.isnan(best["sync"]) and best["sync"] < cfg["SYNC_MIN_AUC"]:
            print(f"[{tag}] NOTE: val sync AUC {best['sync']:.4f} < preregistered "
                  f"SYNC_MIN_AUC {cfg['SYNC_MIN_AUC']}; train sync AUC was {best['train_sync']:.4f}. "
                  f"Train also near chance => still a feature bottleneck; train clearly above "
                  f"chance => a generalisation gap.")

    return {"tag": tag, "seed": seed, "checkpoint_path": ckpt_path, "cfg": cfg,
            "best_epoch": best["epoch"] + 1, "best_val": best,
            "train_idx": train_idx, "val_idx": val_idx, "test_idx": test_idx,
            "train_loader": train_loader, "val_loader": val_loader_l,
            "test_loader": test_loader_l, "model": model_local}


def evaluate_deterministic(model_local, loader, cfg):
    """Single deterministic pass (dropout off) -- used for per-epoch val
    tracking and for Cell 8's ablation ranking. Cell 5's reported numbers
    use the MC-Dropout estimator instead, and calibrate under that same
    estimator."""
    m = unwrap(model_local)
    m.eval()
    acc = defaultdict(list)
    with torch.no_grad():
        for batch in loader:
            crops, mfcc, sync_a, sync_off, vlab, alab, mask, gid = [b.to(device) for b in batch]
            with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                # Backbone once; both the matched and mismatched sync passes
                # below reuse these tokens (see the training loop's note).
                v_tok, m_tok = m.encode_visual(crops)
                out = m.forward_from_tokens(v_tok, m_tok, mfcc, sync_a, visual_mask=mask)
            acc["v_prob"].extend(torch.sigmoid(out["video_logit"].float()).cpu().numpy())
            acc["v_true"].extend(vlab.cpu().numpy())
            acc["a_prob"].extend(torch.sigmoid(out["audio_logit"].float()).cpu().numpy())
            acc["a_true"].extend(alab.cpu().numpy())
            if out["fusion_logit"] is not None:
                acc["f_prob"].extend(torch.sigmoid(out["fusion_logit"].float()).cpu().numpy())
                acc["f_true"].extend(torch.clamp(vlab + alab, 0, 1).cpu().numpy())
            if out["sync_logit"] is not None:
                B = crops.size(0)
                perm = torch.roll(torch.arange(B, device=device), 1)
                # Identity-aware: fall back to the clip's own time-shifted
                # window whenever the rolled partner shares its identity
                # group. v3.3 rolled blindly on an unshuffled loader and so
                # labelled many true matches as mismatched.
                ok = gid[perm] != gid
                neg_a = torch.where(ok.view(-1, 1, 1), sync_a[perm], sync_off)
                with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                    out_neg = m.forward_from_tokens(v_tok, m_tok, mfcc, neg_a, visual_mask=mask)
                acc["s_prob"].extend(torch.sigmoid(out["sync_logit"].float()).cpu().numpy())
                acc["s_prob"].extend(torch.sigmoid(out_neg["sync_logit"].float()).cpu().numpy())
                acc["s_true"].extend([0] * B + [1] * B)
    return {"video_auc": safe_auc(acc["v_true"], acc["v_prob"]),
            "audio_auc": safe_auc(acc["a_true"], acc["a_prob"]),
            "sync_auc": safe_auc(acc["s_true"], acc["s_prob"]) if acc["s_true"] else float('nan'),
            "fusion_auc": safe_auc(acc["f_true"], acc["f_prob"]) if acc["f_true"] else float('nan'),
            "raw": acc}


# ---------------------------------------------------------------------
# Main run: N_SEEDS repeats, mean +/- std
# ---------------------------------------------------------------------
seed_results = []
for si in range(CONFIG["N_SEEDS"]):
    s = CONFIG["SEED"] + si
    res = run_training_pipeline(CONFIG, tag="main", seed=s, verbose=True)
    seed_results.append(res)
    if si < CONFIG["N_SEEDS"] - 1:
        del res["model"]
        torch.cuda.empty_cache()

_main_result = seed_results[-1]
model = _main_result["model"]
checkpoint_path = _main_result["checkpoint_path"]
val_loader = _main_result["val_loader"]
test_loader = _main_result["test_loader"]
train_idx, val_idx, test_idx = (_main_result["train_idx"], _main_result["val_idx"],
                                _main_result["test_idx"])


def _ms(key):
    vals = [r["best_val"][key] for r in seed_results if not np.isnan(r["best_val"][key])]
    return (float(np.mean(vals)), float(np.std(vals))) if vals else (float('nan'), float('nan'))


print("\n" + "=" * 68)
print(f"  MAIN RUN across {CONFIG['N_SEEDS']} seeds -- VALIDATION AUC (mean +/- std)")
print("=" * 68)
for k, name in (("video", "Video head"), ("audio", "Audio head"),
                ("sync", "Sync head "), ("fusion", "Fusion head")):
    m_, s_ = _ms(k)
    print(f"  {name}: {m_:.4f} +/- {s_:.4f}")
gap_m, gap_s = float(np.mean([r["best_val"]["train_video"] - r["best_val"]["video"]
                              for r in seed_results])), 0.0
print(f"  Video train/val gap: {gap_m:+.4f}   (v3.3: +0.2473)")

best_metrics = {
    "n_seeds": CONFIG["N_SEEDS"],
    "per_seed": [{"seed": r["seed"], "best_epoch": r["best_epoch"],
                  **{k: r["best_val"][k] for k in ("video", "audio", "sync", "fusion",
                                                   "train_video", "train_sync")}}
                 for r in seed_results],
    "val_video_auc_mean_std": _ms("video"), "val_audio_auc_mean_std": _ms("audio"),
    "val_sync_auc_mean_std": _ms("sync"), "val_fusion_auc_mean_std": _ms("fusion"),
    "n_train": len(train_idx), "n_val": len(val_idx), "n_test": len(test_idx),
    "config": {k: str(v) for k, v in CONFIG.items()},
}
with open("/kaggle/working/best_metrics_v4.json", "w") as f:
    json.dump(best_metrics, f, indent=2)
print(f"\nCells 5-8 use the last seed's model. Checkpoint: {checkpoint_path}")
print("best_metrics_v4.json written.")

### Cell 5 — Calibration Fit Under the Matched Estimator, Full Report on Test
Fits per-head temperature and bootstrap-median thresholds on VAL using `mc_collect()` — the SAME
20-sample MC-Dropout estimator Pass 3 then applies to TEST. This is the direct fix for Root Cause 2
(v3.3 fit calibration on a single dropout-off pass and applied it to a 20-sample dropout-on average).
`min_recall_floor` is deliberately removed: it was papering over the estimator mismatch, not a real
fix. Reports video/audio/fusion/sync AUC with bootstrap CIs (now honest — v4's test split has zero
pseudo-replication), balanced accuracy alongside plain accuracy, and gates the sync head out of
`calibration.json`'s "reportable" verdict unless its test AUC clears the preregistered
`SYNC_MIN_AUC`.

In [ ]:
# =====================================================================
# CELL 5: CALIBRATION FIT UNDER THE SAME ESTIMATOR USED AT TEST,
#         FULL REPORT (ECE, ROC, ATTRIBUTION MATRIX, BOOTSTRAP CIs)
#         ON THE HELD-OUT TEST SPLIT
#
# THE ROOT-CAUSE-2 FIX THIS CELL EXISTS FOR:
# v3.3's collect_deterministic() fit temperature and both thresholds with
# model.eval() -- dropout OFF, one forward pass. Its Pass 3 then evaluated
# TEST with mc_predict_calibrated(n_samples=20) -- dropout ON, 20 stochastic
# passes averaged. Dropout at p=0.4, propagated through a nonlinear head,
# shifts the logit distribution; calibrating on one distribution and
# applying to another is why v3.3's video ECE was 0.1903 AFTER temperature
# scaling, why its fitted temperature was an implausible 4.301, and why the
# audio threshold needed an arbitrary min_recall_floor=0.5 band-aid to stop
# collapsing.
#
# THE FIX: mc_collect() below is the ONLY estimator used anywhere in this
# cell. Temperature and both thresholds are fit on val's MC-mean logits;
# test is scored with the identical estimator. min_recall_floor is REMOVED
# -- if it turns out to still be needed, that means this fix did not work
# and is a signal to stop, not a threshold to paper over with.
#
# WHAT ELSE CHANGED vs v3.3:
#  * FUSION HEAD reported as a THIRD verdict signal alongside video/audio,
#    not just an auxiliary. It is the payoff of the cross-attention path
#    that ablation A9 (Cell 8) tests fairly.
#  * SYNC HEAD IS GATED: it only appears in the report / calibration.json
#    as a usable verdict signal if its test AUC clears the preregistered
#    CONFIG["SYNC_MIN_AUC"] (0.70). Below that, it is reported for
#    completeness but calibration.json marks it "reportable: false" so
#    Cell 7 does not display a manipulated signal as if it meant anything.
#  * BALANCED ACCURACY reported alongside plain accuracy (video real:fake
#    is 1000:500 in the base pool -- plain accuracy alone can hide a
#    head that just learned the prior).
#  * Bootstrap CIs are now honest: v3.3 pseudo-replicated ~1,500 clips into
#    ~5,000 "samples" via baked augmentation, so its CIs treated ~3x more
#    independent observations than existed. v4's test split has zero
#    pseudo-replication (Cell 3 caches one row per clip), so N here really
#    is the number of independent clips.
#  * VAL vs TEST separation is unchanged from v3.3's correct design:
#    thresholds/temperature/uncertainty cutoffs are fit on val; every
#    reported number is computed on test, touched once.
# =====================================================================

import json
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from sklearn.metrics import (confusion_matrix, roc_curve, auc, classification_report,
                             roc_auc_score, accuracy_score, balanced_accuracy_score)

eval_model = build_model(CONFIG).to(device)
eval_model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=False))
eval_model.eval()


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def mc_collect(model, loader, cfg, n_samples=20, need_sync=True, need_fusion=True):
    """MC-Dropout collection over a full loader -- the ONE estimator used
    for calibration fitting AND for every reported number, closing the
    mismatch that produced v3.3's ECE 0.19 / temperature 4.3.

    Returns per-head dicts of {mean_logit, std_logit (uncertainty), true}
    as flat numpy arrays over every sample in the loader. Sync negatives
    use the same identity-aware construction as Cell 4's
    evaluate_deterministic (audio from a different identity group, else
    the clip's own time-shifted window) -- never a blind roll on an
    unshuffled loader, which is what mislabelled true matches as
    "mismatched" in v3.3."""
    m = unwrap(model)
    m.eval()
    out = defaultdict(list)
    with torch.no_grad():
        for batch in loader:
            crops, mfcc, sync_a, sync_off, vlab, alab, mask, gid = [b.to(device) for b in batch]
            # Backbone once per batch, shared across the matched and (below)
            # mismatched sync scoring -- otherwise the whole EfficientNet runs
            # twice on identical pixels for every batch of both val and test.
            v_tok, m_tok = m.encode_visual(crops)
            mc = mc_logits(m, crops, mfcc, sync_a, visual_mask=mask, n_samples=n_samples,
                           tokens=(v_tok, m_tok))
            out["video_logit"].append(mc["video"]); out["video_true"].append(vlab.cpu().numpy())
            out["audio_logit"].append(mc["audio"]); out["audio_true"].append(alab.cpu().numpy())
            if need_fusion and "fusion" in mc:
                out["fusion_logit"].append(mc["fusion"])
                out["fusion_true"].append(torch.clamp(vlab + alab, 0, 1).cpu().numpy())
            if need_sync and "sync" in mc:
                B = crops.size(0)
                perm = torch.roll(torch.arange(B, device=device), 1)
                ok = gid[perm] != gid
                neg_a = torch.where(ok.view(-1, 1, 1), sync_a[perm], sync_off)
                mc_neg = mc_logits(m, crops, mfcc, neg_a, visual_mask=mask, n_samples=n_samples,
                                   tokens=(v_tok, m_tok))
                out["sync_logit"].append(np.concatenate([mc["sync"], mc_neg["sync"]], axis=1))
                out["sync_true"].append(np.concatenate([np.zeros(B), np.ones(B)]))

    result = {}
    for head in ("video", "audio", "fusion", "sync"):
        if f"{head}_logit" not in out:
            continue
        logits = np.concatenate(out[f"{head}_logit"], axis=1)     # (n_samples, N)
        true = np.concatenate(out[f"{head}_true"])
        result[head] = {"mean_logit": logits.mean(axis=0), "std_prob": sigmoid(logits).std(axis=0),
                        "true": true.astype(int)}
    return result


# ---- Pass 1: MC-collect VAL, fit temperature + thresholds on THAT -------
print(f"Collecting MC-Dropout logits on VAL ({CONFIG['N_MC_SAMPLES']} passes/sample, "
      f"the SAME estimator Pass 3 below applies to TEST)...")
val_mc = mc_collect(eval_model, val_loader, CONFIG, n_samples=CONFIG["N_MC_SAMPLES"])


def fit_temperature(mean_logits, labels, max_iter=100, lr=0.05):
    if len(set(labels.astype(int).tolist())) < 2:
        return 1.0
    logits_t = torch.tensor(mean_logits, dtype=torch.float32)
    labels_t = torch.tensor(labels, dtype=torch.float32)
    T = torch.nn.Parameter(torch.ones(1))
    opt = torch.optim.LBFGS([T], lr=lr, max_iter=max_iter)
    bce = nn.BCEWithLogitsLoss()

    def closure():
        opt.zero_grad()
        loss = bce(logits_t / T.clamp(min=1e-2), labels_t)
        loss.backward()
        return loss

    opt.step(closure)
    return float(T.clamp(min=1e-2).item())


def bootstrap_stable_threshold(y_true, y_prob, n_boot=200, seed=42, default=0.5):
    """Median Youden's-J threshold across n_boot bootstrap resamples of val
    -- a single point estimate is a high-variance estimator for a modest,
    possibly skewed val set. min_recall_floor is DELIBERATELY REMOVED (see
    header): once calibration and application match, the recall collapse
    that floor was papering over should not recur. If it does, that is a
    signal to re-open Root Cause 2, not to re-add the floor."""
    y_true, y_prob = np.asarray(y_true), np.asarray(y_prob)
    if len(set(y_true.astype(int).tolist())) < 2:
        return default, np.array([])
    rng = np.random.RandomState(seed)
    n = len(y_true)
    thresholds = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        yt, yp = y_true[idx], y_prob[idx]
        if len(set(yt.astype(int).tolist())) < 2:
            continue
        fpr, tpr, thr = roc_curve(yt, yp)
        thresholds.append(float(thr[np.argmax(tpr - fpr)]))
    if not thresholds:
        return default, np.array([])
    thresholds = np.array(thresholds)
    return float(np.median(thresholds)), thresholds


temps, thrs, thr_boot = {}, {}, {}
for head in ("video", "audio", "fusion", "sync"):
    if head not in val_mc:
        continue
    temps[head] = fit_temperature(val_mc[head]["mean_logit"], val_mc[head]["true"])
    p = sigmoid(val_mc[head]["mean_logit"] / temps[head])
    thrs[head], thr_boot[head] = bootstrap_stable_threshold(val_mc[head]["true"], p)

print(f"Fitted temperatures (on MC-mean val logits) -> " +
      " | ".join(f"{h}: {t:.3f}" for h, t in temps.items()))
print(f"Bootstrap-median thresholds (val, {CONFIG['N_MC_SAMPLES']}-sample MC estimator, "
      f"{len(next(iter(thr_boot.values()), []))} resamples) -> " +
      " | ".join(f"{h}: {t:.4f}" for h, t in thrs.items()))
for h in thr_boot:
    if len(thr_boot[h]):
        print(f"  {h} threshold stability (bootstrap std, lower = more stable): "
              f"{thr_boot[h].std():.4f}")

unc_flag = {h: float(np.percentile(val_mc[h]["std_prob"], 95)) for h in val_mc}
print(f"Uncertainty-flag cutoffs (95th pct of VAL MC-Dropout std) -> " +
      " | ".join(f"{h}: {v:.4f}" for h, v in unc_flag.items()))

sync_reportable = ("sync" in val_mc) and (thrs.get("sync") is not None)

# ---- Pass 2: MC-collect TEST -- touched here for the first/only time ----
print(f"\nCollecting MC-Dropout logits on TEST (identical estimator)...")
test_mc = mc_collect(eval_model, test_loader, CONFIG, n_samples=CONFIG["N_MC_SAMPLES"])


def expected_calibration_error(probs, labels, n_bins=15):
    """Standard binary ECE: gap between mean predicted confidence and the
    empirical FRACTION OF POSITIVES per bin."""
    bins = np.linspace(0, 1, n_bins + 1)
    ece, bin_fracs, bin_confs = 0.0, [], []
    for i in range(n_bins):
        lo = bins[i] if i > 0 else -0.01
        m = (probs > lo) & (probs <= bins[i + 1])
        if m.sum() > 0:
            bin_frac, bin_conf = labels[m].mean(), probs[m].mean()
            ece += (m.sum() / len(probs)) * abs(bin_frac - bin_conf)
            bin_fracs.append(bin_frac); bin_confs.append(bin_conf)
        else:
            bin_fracs.append(np.nan); bin_confs.append(np.nan)
    return ece, bin_fracs, bin_confs


def bootstrap_ci(y_true, y_score, metric_fn, n_boot=2000, seed=42):
    rng = np.random.RandomState(seed)
    y_true, y_score = np.asarray(y_true), np.asarray(y_score)
    n = len(y_true)
    stats = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        yt, ys = y_true[idx], y_score[idx]
        if len(set(yt.tolist())) < 2:
            continue
        try:
            stats.append(metric_fn(yt, ys))
        except ValueError:
            continue
    if not stats:
        return float('nan'), float('nan'), float('nan')
    return float(np.mean(stats)), float(np.percentile(stats, 2.5)), float(np.percentile(stats, 97.5))


test_probs, test_preds, test_true = {}, {}, {}
for head in val_mc:
    p = sigmoid(test_mc[head]["mean_logit"] / temps[head])
    test_probs[head] = p
    test_preds[head] = (p >= thrs[head]).astype(int)
    test_true[head] = test_mc[head]["true"]

print("\n" + "=" * 68)
print("  CROSSFUSE v4 TEST-SET REPORT (calibrated, thresholds fit on VAL, "
      "matched MC estimator)")
print("  " + " | ".join(f"{h.capitalize()} thr={thrs[h]:.4f} (T={temps[h]:.2f})" for h in thrs))
print("=" * 68)

head_labels = {"video": ("Real vid", "Fake vid"), "audio": ("Real aud", "Fake aud"),
              "fusion": ("Real clip", "Fake clip"), "sync": ("Matched", "Mismatched")}
ece_all, bacc_curve, aucs_all = {}, {}, {}
for head in ("video", "audio", "fusion", "sync"):
    if head not in test_probs:
        continue
    yt, p, pred = test_true[head], test_probs[head], test_preds[head]
    name = head.upper()
    print(f"\n--- {name} HEAD ---")
    print(classification_report(yt, pred, target_names=list(head_labels[head]), zero_division=0))
    acc = accuracy_score(yt, pred)
    bacc = balanced_accuracy_score(yt, pred) if len(set(yt.tolist())) > 1 else float('nan')
    a_mean, a_lo, a_hi = bootstrap_ci(yt, p, roc_auc_score)
    aucs_all[head] = (a_mean, a_lo, a_hi)
    ece, bf, bc = expected_calibration_error(p, yt)
    ece_all[head] = ece
    bacc_curve[head] = (bf, bc)
    print(f"AUC (test, bootstrap 95% CI, n={len(yt)}): {a_mean:.4f} [{a_lo:.4f}, {a_hi:.4f}]")
    print(f"Accuracy: {acc:.4f} | Balanced accuracy: {bacc:.4f} | ECE: {ece:.4f}")
    if head == "video":
        print(f"Mean MC-Dropout uncertainty: {test_mc[head]['std_prob'].mean():.4f}")

if "sync" in test_probs:
    s_auc = aucs_all["sync"][0]
    if s_auc < CONFIG["SYNC_MIN_AUC"]:
        print(f"\n[!] Sync AUC {s_auc:.4f} < preregistered SYNC_MIN_AUC "
              f"{CONFIG['SYNC_MIN_AUC']} -- sync will NOT be shown as a verdict signal in Cell 7, "
              f"and ablation A9's title decision does not get help from A10 either.")
        sync_reportable = False
    else:
        print(f"\nSync AUC {s_auc:.4f} clears SYNC_MIN_AUC {CONFIG['SYNC_MIN_AUC']} -- "
              f"cross-attention is load-bearing for a genuine AV-correspondence signal.")
        sync_reportable = True

calibration = {
    "temp": temps, "thr": thrs, "unc_flag": unc_flag,
    "sync_reportable": bool(sync_reportable),
    "sync_min_auc": CONFIG["SYNC_MIN_AUC"],
    "test_auc": {h: aucs_all[h][0] for h in aucs_all},
    "n_train": len(train_idx), "n_val": len(val_idx), "n_test": len(test_idx),
}
with open("/kaggle/working/calibration.json", "w") as f:
    json.dump(calibration, f, indent=2)
print("\ncalibration.json written.")

# ---- Per-head plots (test set) ------------------------------------------
plot_heads = [h for h in ("video", "audio", "fusion") if h in test_probs]
sns.set_theme(style="whitegrid", font_scale=1.0)
fig, axes = plt.subplots(len(plot_heads), 3, figsize=(18, 5 * len(plot_heads)))
if len(plot_heads) == 1:
    axes = axes.reshape(1, 3)
for row, head in enumerate(plot_heads):
    yt, p, pred, thr = test_true[head], test_probs[head], test_preds[head], thrs[head]
    ece, bf, bc = ece_all[head], *bacc_curve[head]
    cm = confusion_matrix(yt, pred, labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[row, 0],
               xticklabels=["Pred real", "Pred fake"], yticklabels=["Actual real", "Actual fake"],
               annot_kws={"size": 14})
    axes[row, 0].set_title(f"{head.capitalize()} -- Confusion (test, p>={thr:.2f})",
                           fontsize=12, fontweight='bold')

    if len(set(int(v) for v in yt)) > 1:
        fpr, tpr, _ = roc_curve(yt, p)
        oi = np.argmax(tpr - fpr)
        axes[row, 1].plot(fpr, tpr, color='#1f77b4', lw=3,
                          label=f'{head.capitalize()} (AUC={aucs_all[head][0]:.3f})')
        axes[row, 1].scatter(fpr[oi], tpr[oi], color='red', s=90, zorder=5, label=f'Cutoff {thr:.2f}')
    axes[row, 1].plot([0, 1], [0, 1], color='gray', lw=1.2, linestyle='--')
    axes[row, 1].set_xlabel('FPR'); axes[row, 1].set_ylabel('TPR')
    axes[row, 1].set_title(f'{head.capitalize()} -- ROC (test)', fontsize=12, fontweight='bold')
    axes[row, 1].legend(loc="lower right")

    axes[row, 2].plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
    axes[row, 2].plot(bc, bf, marker='o', color='#d62728', label=f'{head.capitalize()} (ECE={ece:.3f})')
    axes[row, 2].set_xlabel('Mean predicted confidence')
    axes[row, 2].set_ylabel('Empirical fraction of positives')
    axes[row, 2].set_title(f'{head.capitalize()} -- Reliability (test)', fontsize=12, fontweight='bold')
    axes[row, 2].legend(loc="upper left")

plt.tight_layout()
plt.savefig("/kaggle/working/crossfuse_v4_perhead_plots.png", dpi=300, bbox_inches='tight')
plt.show()

if "sync" in test_probs:
    fpr_s, tpr_s, _ = roc_curve(test_true["sync"], test_probs["sync"])
    plt.figure(figsize=(6, 5))
    plt.plot(fpr_s, tpr_s, color='#2ca02c', lw=3, label=f'Sync (AUC={aucs_all["sync"][0]:.3f})')
    plt.plot([0, 1], [0, 1], color='gray', lw=1.2, linestyle='--')
    plt.xlabel('FPR'); plt.ylabel('TPR')
    plt.title('AV Sync Head -- ROC (test, identity-aware negatives)', fontsize=13, fontweight='bold')
    plt.legend(loc="lower right"); plt.tight_layout()
    plt.savefig("/kaggle/working/crossfuse_v4_sync_roc.png", dpi=300, bbox_inches='tight')
    plt.show()

# ---- 4-way modality-attribution confusion (video x audio, test set) -----
def to_category_idx(vf, af):
    return {(0, 0): 0, (0, 1): 1, (1, 0): 2, (1, 1): 3}[(int(vf), int(af))]

cat_names = ["RVRA\n(real/real)", "RVFA\n(real/fakeAud)", "FVRA\n(fakeVid/real)", "FVFA\n(fake/fake)"]
true_cat = [to_category_idx(vt, at) for vt, at in zip(test_true["video"], test_true["audio"])]
pred_cat = [to_category_idx(vp, ap) for vp, ap in zip(test_preds["video"], test_preds["audio"])]
cm4 = confusion_matrix(true_cat, pred_cat, labels=[0, 1, 2, 3])

plt.figure(figsize=(7.5, 6))
sns.heatmap(cm4, annot=True, fmt='d', cmap='Greens', cbar=False,
           xticklabels=cat_names, yticklabels=cat_names, annot_kws={"size": 12})
plt.title("Modality Attribution -- Derived 4-Category Confusion (test)", fontsize=13,
         fontweight='bold', pad=12)
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.tight_layout()
plt.savefig("/kaggle/working/crossfuse_v4_attribution_matrix.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"\nThresholds (val) -> " + " | ".join(f"{h}: {thrs[h]:.4f}" for h in thrs))
print("All classification reports, ECE, ROC/reliability plots and the attribution matrix above "
      "are computed on the TEST split, touched here for the first and only time, with the SAME "
      "MC-Dropout estimator used to fit calibration.")

### Cell 6 — Cross-Dataset Generalization: Celeb-DF v2 (headline) + DFDC (appendix)
v3.3's DFDC benchmark was capped at 77 real clips, giving a bootstrap CI that included chance
(0.5821 [0.4902, 0.6706]) — a null result, not a headline. Celeb-DF v2's official
`List_of_testing_videos.txt` (518 clips) replaces it as the primary cross-dataset number; DFDC is kept
as a small, clearly-labelled, ACTUALLY COMPUTED appendix through the same extraction/scoring path
(`extract_and_score_video_only`), rather than hidden. Retains v3.3's stale-cache purge, a real bug it
caught (54 stale tensors once inflated an eval set from 77/77 to 77/131).

In [ ]:
# =====================================================================
# CELL 6: CROSS-DATASET GENERALIZATION -- CELEB-DF v2 ZERO-SHOT (VIDEO HEAD)
#
# WHY THIS REPLACES v3.3's DFDC CELL:
# v3.3's DFDC benchmark was structurally capped at `min(300, len(reals),
# len(fakes))` and the attached DFDC sample only had 77 real clips, so its
# headline (AUC 0.5821, bootstrap 95% CI [0.4902, 0.6706]) INCLUDES CHANCE.
# A CI that contains 0.5 is a null result, not a generalization number, and
# cannot be a paper headline.
#
# Celeb-DF v2's OFFICIAL List_of_testing_videos.txt (518 clips) is used
# here instead: it is video-only (Celeb-DF audio is genuine, matching what
# this head actually attributes), it is THE standard cross-dataset
# benchmark in this literature (numbers here are directly comparable to
# published work), and 518 clips gives a CI roughly half the width of
# DFDC's 154-clip sample.
#
# DFDC is kept below as an OPTIONAL APPENDIX through the SAME extraction/
# scoring path (extract_and_score_video_only), clearly labelled with its
# small n and (typically) chance-inclusive CI -- reviewers ask about DFDC
# even when Celeb-DF is the headline, so it is worth actually computing as
# a footnote rather than hand-waved away.
#
# RETAINED from v3.3: the stale-cached-tensor purge (a real bug it caught:
# 54 stale tensors once silently inflated an eval set from 77/77 to
# 77/131), same ffmpeg/MTCNN pipeline as training, make_rng(tag) for a
# reproducible sample draw independent of how many other random calls
# happened earlier in the run.
# =====================================================================

import os
import json as _json
import glob
import hashlib
import numpy as np
import torch
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from sklearn.metrics import roc_curve, classification_report

BASE_CACHE = "/kaggle/working/xdataset_crops_v4"
os.makedirs(BASE_CACHE, exist_ok=True)


class VideoOnlyCropDataset(torch.utils.data.Dataset):
    """Centre-window, no augmentation -- identical eval-time semantics to
    CrossFuseCropDataset's eval branch (Cell 4), so a cross-dataset AUC is
    comparable to the in-dataset test AUC. Audio is a zero dummy: both
    Celeb-DF and DFDC's fake audio is (predominantly) genuine, and this
    cell reports the VIDEO head only."""

    def __init__(self, rows, cache_dir, window_frames):
        self.rows, self.cache_dir, self.W = rows, cache_dir, window_frames

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        crops = np.load(os.path.join(self.cache_dir, r["key"] + ".npy"), mmap_mode="r")
        meta = np.load(os.path.join(self.cache_dir, r["key"] + "_meta.npz"))
        valid = meta["valid"]
        T = crops.shape[0]
        W = min(self.W, T)
        start = max(0, T - W) // 2
        window = np.array(crops[start:start + W])
        win_valid = np.array(valid[start:start + W], dtype=bool)
        if not win_valid.any():
            win_valid[0] = True
        pad = self.W - W
        if pad > 0:
            window = np.concatenate([window, np.zeros((pad, *window.shape[1:]), np.uint8)], 0)
            win_valid = np.concatenate([win_valid, np.zeros(pad, bool)])
        dummy_mfcc = np.zeros((40, CONFIG["AUDIO_FRAMES"]), dtype=np.float32)
        return (torch.from_numpy(np.ascontiguousarray(window)), torch.from_numpy(dummy_mfcc),
               torch.tensor(float(r["video_label"])), torch.from_numpy(win_valid))


def _process_one(video_path, cache_dir, key, mtcnn_local):
    """Per-clip extraction, wrapped in try/except -- a single corrupt/
    unreadable clip out of 518+ (Celeb-DF) or ~1000 (DFDC) must not crash
    the whole cell via an unhandled exception surfacing through
    fut.result() in the main thread. Same failure-tolerance design as
    Cell 3's process_job."""
    crop_path = os.path.join(cache_dir, key + ".npy")
    meta_path = os.path.join(cache_dir, key + "_meta.npz")
    if os.path.exists(crop_path) and os.path.exists(meta_path):
        return "cached"
    try:
        rng = make_rng(f"xdataset:{key}")
        crops, valid, fps, start, ts = extract_face_crops_contiguous(video_path, mtcnn_local, rng=rng)
        if crops is None or valid.sum() == 0:
            return "no_faces"
        np.save(crop_path, crops)
        np.savez(meta_path, valid=valid, timestamps=ts)
        return "ok"
    except Exception as e:
        return f"error:{type(e).__name__}"


def _xdataset_key(key_prefix, video_path):
    """Collision-proof key: basename-before-first-dot is usually unique for
    Celeb-DF/DFDC filenames, but a short path hash is appended anyway (same
    collision-proofing philosophy as Cell 3's make_sample_key) so two clips
    that happened to share a prefix can never silently overwrite each
    other's cache entry."""
    stem = os.path.basename(video_path).split('.')[0]
    h = hashlib.md5(video_path.replace("\\", "/").encode()).hexdigest()[:8]
    return f"{key_prefix}_{stem}_{h}"


def extract_and_score_video_only(entries, key_prefix, desc, min_per_class_for_headline=150):
    """entries: list of (video_path, label) with label 1=fake, 0=real.
    Shared by Celeb-DF (headline) and DFDC (appendix) -- one extraction/
    scoring path so the two benchmarks cannot silently drift apart the way
    v3.3's hand-rolled DFDC loop drifted from its training pipeline."""
    cache_dir = os.path.join(BASE_CACHE, key_prefix)
    os.makedirs(cache_dir, exist_ok=True)

    # Purge stale cached crops left over from a DIFFERENT prior sample --
    # the exact v3.3 bug (54 stale tensors inflated 77/77 to 77/131).
    expected = {_xdataset_key(key_prefix, p) for p, _ in entries}
    existing = {f[:-4] for f in os.listdir(cache_dir) if f.endswith(".npy") and "_meta" not in f}
    stale = existing - expected
    if stale:
        print(f"  Removing {len(stale)} stale cached crop(s) from a previous {key_prefix} run...")
        for key in stale:
            for suffix in (".npy", "_meta.npz"):
                p = os.path.join(cache_dir, key + suffix)
                if os.path.exists(p):
                    os.remove(p)

    if 'detectors' not in globals():
        print("  [i] Building fresh face detectors (Cell 3 didn't run in this session).")
        globals()['detectors'] = {d: build_detector(d) for d in DEVICE_IDS}

    rows, results_log = [], defaultdict(int)
    with ThreadPoolExecutor(max_workers=len(DEVICE_IDS)) as pool:
        futures = {}
        for i, (v, lbl) in enumerate(entries):
            key = _xdataset_key(key_prefix, v)
            dstr = DEVICE_IDS[i % len(DEVICE_IDS)]
            futures[pool.submit(_process_one, v, cache_dir, key, detectors[dstr])] = (key, lbl)
        for fut in tqdm(as_completed(futures), total=len(futures), desc=desc):
            key, lbl = futures[fut]
            status = fut.result()
            results_log[status] += 1
            if status in ("ok", "cached"):
                rows.append({"key": key, "video_label": lbl})
    print(f"  {desc}: {dict(results_log)} -> {len(rows)} usable "
         f"({sum(1 for r in rows if r['video_label'] == 0)} real / "
         f"{sum(1 for r in rows if r['video_label'] == 1)} fake)")

    n_real = sum(1 for r in rows if r["video_label"] == 0)
    n_fake = sum(1 for r in rows if r["video_label"] == 1)
    if min(n_real, n_fake) < min_per_class_for_headline:
        print(f"  [!] Only {min(n_real, n_fake)} clips in the smaller class -- "
              f"CI will be wide{'; treat as a footnote, not a headline' if min(n_real, n_fake) < 50 else ''}.")

    loader = torch.utils.data.DataLoader(
        VideoOnlyCropDataset(rows, cache_dir, CONFIG["WINDOW_FRAMES"]),
        batch_size=CONFIG["BATCH_CLIPS"], shuffle=False, num_workers=CONFIG["NUM_WORKERS"])

    m_eval = unwrap(eval_model)
    m_eval.eval()
    true_all, mean_logit_all = [], []
    with torch.no_grad():
        for crops, mfcc, vlab, mask in tqdm(loader, desc=f"Scoring {desc} (MC-Dropout)"):
            crops, mfcc, mask = crops.to(device), mfcc.to(device), mask.to(device)
            mc = mc_logits(m_eval, crops, mfcc, sync_audio=None, visual_mask=mask,
                           n_samples=CONFIG["N_MC_SAMPLES"])
            mean_logit_all.append(mc["video"].mean(axis=0))
            true_all.append(vlab.numpy())
    true_all = np.concatenate(true_all).astype(int)
    mean_logit_all = np.concatenate(mean_logit_all)
    prob_all = sigmoid(mean_logit_all / temps["video"])
    return true_all, prob_all


def report_video_generalization(name, true, prob, headline=True):
    auc_mean, auc_lo, auc_hi = bootstrap_ci(true, prob, roc_auc_score)
    pred_transferred = (prob >= thrs["video"]).astype(int)
    acc_t = accuracy_score(true, pred_transferred)
    bacc_t = balanced_accuracy_score(true, pred_transferred) if len(set(true.tolist())) > 1 else float('nan')
    if len(set(true.tolist())) > 1:
        fpr_o, tpr_o, thr_o = roc_curve(true, prob)
        oracle_thr = float(thr_o[np.argmax(tpr_o - fpr_o)])
        acc_oracle = accuracy_score(true, (prob >= oracle_thr).astype(int))
    else:
        oracle_thr, acc_oracle = float('nan'), float('nan')
    ece, _, _ = expected_calibration_error(prob, true)
    val_video_auc = aucs_all["video"][0]
    gen_gap = val_video_auc - auc_mean

    tag = "HEADLINE" if headline else "APPENDIX -- footnote only"
    print(f"\n{'=' * 68}\n  {name} ZERO-SHOT REPORT -- VIDEO HEAD [{tag}] "
          f"(n={len(true)}, thr={thrs['video']:.4f})\n{'=' * 68}")
    print(classification_report(true, pred_transferred, target_names=["Real", "Fake"], zero_division=0))
    print(f">> {name} Zero-Shot AUC (bootstrap 95% CI): {auc_mean:.4f} [{auc_lo:.4f}, {auc_hi:.4f}]")
    print(f">> Zero-Shot Accuracy (transferred thr): {acc_t:.4f} | Balanced accuracy: {bacc_t:.4f}")
    print(f">> Zero-Shot ECE (post-temperature): {ece:.4f}")
    print(f">> [upper bound] Accuracy at {name}-oracle threshold {oracle_thr:.4f}: {acc_oracle:.4f}")
    print(f">> Generalization gap (Val AUC {val_video_auc:.4f} - {name} AUC {auc_mean:.4f}): {gen_gap:+.4f}")
    if auc_lo <= 0.5:
        print(f"[!] CI includes chance -- {name} is a NULL RESULT here, report it as such, "
              f"not as a headline.")

    fpr_c, tpr_c, _ = roc_curve(true, prob)
    plt.figure(figsize=(6.5, 5.5))
    plt.plot(fpr_c, tpr_c, color='#9467bd' if headline else '#7f7f7f', lw=3,
            label=f'{name} (AUC={auc_mean:.3f})')
    plt.plot([0, 1], [0, 1], color='gray', lw=1.2, linestyle='--')
    plt.xlabel('FPR'); plt.ylabel('TPR')
    plt.title(f'Video Head -- {name} Generalization [{tag}]\n'
             f'AUC 95% CI=[{auc_lo:.3f}, {auc_hi:.3f}]', fontsize=12, fontweight='bold')
    plt.legend(loc="lower right"); plt.tight_layout()
    plt.savefig(f"/kaggle/working/crossfuse_v4_{name.lower().replace(' ', '').replace('-', '')}_roc.png",
               dpi=300, bbox_inches='tight')
    plt.show()

    return {"n": len(true), "n_real": int((true == 0).sum()), "n_fake": int((true == 1).sum()),
            "auc_mean": auc_mean, "auc_ci": [auc_lo, auc_hi], "accuracy_transferred": acc_t,
            "balanced_accuracy_transferred": bacc_t, "ece": ece, "oracle_threshold": oracle_thr,
            "accuracy_oracle": acc_oracle, "val_video_auc": val_video_auc,
            "generalization_gap": gen_gap, "headline": headline}


# ---- Celeb-DF v2: the headline cross-dataset benchmark ------------------
def find_celebdf_root():
    candidates = glob.glob("/kaggle/input/**/List_of_testing_videos.txt", recursive=True)
    return os.path.dirname(candidates[0]) if candidates else None


def load_celebdf_test_list(root):
    """Official protocol: each line is '<label> <relative/path.mp4>', label
    1=real, 0=fake. Folder name is a cross-check (Celeb-synthesis is always
    fake; Celeb-real/YouTube-real are always real), so a malformed line
    degrades to a loud skip instead of a silent mislabel."""
    entries = []
    with open(os.path.join(root, "List_of_testing_videos.txt")) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 2:
                continue
            label_str, rel_path = parts
            full_path = os.path.join(root, rel_path)
            if not os.path.exists(full_path):
                hits = glob.glob(os.path.join(root, "**", os.path.basename(rel_path)), recursive=True)
                if not hits:
                    continue
                full_path = hits[0]
            label, folder_says_fake = int(label_str), "synthesis" in rel_path.lower()
            if (label == 0) != folder_says_fake:
                print(f"  [!] Label/folder mismatch for {rel_path} (label={label}) -- skipping")
                continue
            entries.append((full_path, 1 - label))     # convert to fake=1 convention
    return entries


celebdf_root = find_celebdf_root()
if celebdf_root is None:
    print("[!] Celeb-DF v2 not found under /kaggle/input (looking for "
          "List_of_testing_videos.txt). Attach the Celeb-DF v2 Kaggle dataset "
          "and re-run this cell. Skipping the cross-dataset benchmark.")
    celebdf_results = None
else:
    print(f"Celeb-DF v2 root: {celebdf_root}")
    entries = load_celebdf_test_list(celebdf_root)
    reals, fakes = [e for e in entries if e[1] == 0], [e for e in entries if e[1] == 1]
    print(f"Official test list: {len(reals)} real | {len(fakes)} fake "
         f"({len(entries)} total, expected 518)")
    rng_cdf = make_rng("celebdf")
    rng_cdf.shuffle(reals); rng_cdf.shuffle(fakes)
    selected = reals + fakes
    rng_cdf.shuffle(selected)

    cdf_true, cdf_prob = extract_and_score_video_only(selected, "cdf", "Celeb-DF v2")
    celebdf_results = report_video_generalization("Celeb-DF v2", cdf_true, cdf_prob, headline=True)
    with open("/kaggle/working/celebdf_results_v4.json", "w") as f:
        json.dump(celebdf_results, f, indent=2)
    print("celebdf_results_v4.json written.")

# ---- DFDC: kept as a real, computed, but clearly-labelled APPENDIX ------
def load_dfdc_ground_truth(max_per_class=150, rng=None):
    labeled = []
    for meta_path in sorted(glob.glob("/kaggle/input/**/metadata.json", recursive=True)):
        folder = os.path.dirname(meta_path)
        with open(meta_path) as f:
            data = _json.load(f)
        for vid_name, info in data.items():
            full_path = os.path.join(folder, vid_name)
            if not os.path.exists(full_path):
                continue
            lbl = info.get('label', '').upper()
            if lbl in ('REAL', 'FAKE'):
                labeled.append((full_path, 0 if lbl == 'REAL' else 1))
    reals, fakes = [x for x in labeled if x[1] == 0], [x for x in labeled if x[1] == 1]
    if rng is not None:
        rng.shuffle(reals); rng.shuffle(fakes)
    n = min(max_per_class, len(reals), len(fakes))
    if n < max_per_class:
        print(f"  [!] Only {n} DFDC clips/class available (requested {max_per_class}) -- "
             f"CI will be wide.")
    return reals[:n] + fakes[:n]


dfdc_entries = load_dfdc_ground_truth(rng=make_rng("dfdc_appendix"))
if len(dfdc_entries) < 20:
    print(f"\n[i] DFDC appendix skipped -- only {len(dfdc_entries)} clips found "
          f"(attach a DFDC sample dataset to include this footnote).")
    dfdc_results = None
else:
    print(f"\n[APPENDIX] DFDC: {len(dfdc_entries)} clips found "
         f"({sum(1 for _, l in dfdc_entries if l == 0)} real / "
         f"{sum(1 for _, l in dfdc_entries if l == 1)} fake)")
    dfdc_true, dfdc_prob = extract_and_score_video_only(dfdc_entries, "dfdc", "DFDC")
    dfdc_results = report_video_generalization("DFDC", dfdc_true, dfdc_prob, headline=False)
    with open("/kaggle/working/dfdc_results_v4.json", "w") as f:
        json.dump(dfdc_results, f, indent=2)
    print("dfdc_results_v4.json written (appendix only -- do not lead the paper with this number).")

### Cell 6b — Cross-Generator Audio Zero-Shot: ASVspoof 2019 LA
The audio head's 0.997 in-dataset AUC is real, but every fake in FakeAVCeleb comes from ONE TTS/VC
system — identity-disjoint splitting does nothing to protect against a generator fingerprint, since
the same generator produces every fake in every split. ASVspoof 2019 LA's eval partition uses six
synthesis/conversion systems absent from its own train/dev and from FakeAVCeleb entirely: a genuine
zero-shot test of "synthetic speech" vs. "this one generator". Audio-only (`predict_audio_only()`
calls just the audio branch — no face detection needed). The confound is stated up front: ASVspoof is
studio-condition speech vs. FakeAVCeleb's YouTube audio, so this conflates generator shift with domain
shift — treat the AUC as an upper bound on transfer failure, and a large drop as a legitimate,
publishable negative finding rather than a flaw to hide.

In [ ]:
# =====================================================================
# CELL 6b: CROSS-GENERATOR AUDIO ZERO-SHOT -- ASVspoof 2019 LA
#
# WHY THIS CELL EXISTS:
# The audio head's 0.997 in-dataset AUC is a real number, but ALL fake
# audio in FakeAVCeleb comes from ONE TTS/VC system. Identity-disjoint
# splitting (Cell 4) protects against identity leakage; it does nothing
# about generator-fingerprint leakage, because the SAME generator produces
# every fake in train, val AND test. A classifier that keyed entirely on
# that one generator's fingerprint would still show 0.997 here. This is
# the single biggest attack surface a reviewer has on this paper, so it
# is tested directly rather than left as a caveat in the text.
#
# ASVspoof 2019 LA's eval partition uses SIX synthesis/conversion systems
# (A16-A19 TTS, A17-A19 VC, etc.) that never appear in its own train/dev
# partitions, let alone in FakeAVCeleb -- a genuine zero-shot test of
# whether the audio head learned "synthetic speech" or "this one
# generator". Audio-only, so it needs no face detection: this cell is a
# tiny fraction of Celeb-DF's extraction cost.
#
# THE CONFOUND, STATED UP FRONT (put this in the paper, don't bury it):
# ASVspoof is close-mic'd, studio-condition speech; FakeAVCeleb's audio is
# YouTube-sourced. A drop here conflates GENERATOR shift with DOMAIN
# (recording-condition) shift -- we cannot fully separate the two with a
# single external set. Treat the AUC as an upper bound on how well the
# generator-specific fingerprint theory would fail in the wild, not as a
# clean ablation of generator identity alone.
#
# Audio-only forward pass bypasses the visual backbone entirely (no crops,
# no MTCNN) -- predict_audio_only() below calls just the audio branch of
# CrossFuseModelV4, which is the whole reason the model's audio_encoder/
# audio_head were kept as a clean, independently-callable submodule.
# =====================================================================

import os
import glob
import numpy as np
import torch
from tqdm import tqdm
from sklearn.metrics import roc_curve, classification_report


def find_asvspoof_eval():
    """Returns (protocol_path, flac_dir) for the ASVspoof2019 LA eval
    partition, or (None, None) if not attached. Kaggle mirrors of this
    dataset vary in nesting depth, so we glob broadly for the protocol
    file and infer the flac directory from sibling folders."""
    proto_candidates = (
        glob.glob("/kaggle/input/**/ASVspoof2019.LA.cm.eval.trl.txt", recursive=True) +
        glob.glob("/kaggle/input/**/*LA.cm.eval.trl.txt", recursive=True))
    if not proto_candidates:
        return None, None
    proto_path = proto_candidates[0]
    root = os.path.dirname(os.path.dirname(proto_path))   # .../ASVspoof2019_LA_cm_protocols/..
    flac_candidates = (
        glob.glob(os.path.join(os.path.dirname(proto_path), "..", "**", "ASVspoof2019_LA_eval",
                              "flac"), recursive=True) +
        glob.glob("/kaggle/input/**/ASVspoof2019_LA_eval/flac", recursive=True))
    flac_dir = flac_candidates[0] if flac_candidates else None
    return proto_path, flac_dir


def load_asvspoof_eval_list(proto_path, flac_dir, max_per_class=1000, rng=None):
    """Protocol line format: 'speaker_id filename - attack_id label', label
    in {'bonafide', 'spoof'}. Returns [(flac_path, label)], label 1=spoof
    (fake), 0=bonafide (real) -- the same fake=1 convention used
    everywhere else in this notebook."""
    reals, fakes = [], []
    with open(proto_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            filename, label_str = parts[1], parts[-1]
            flac_path = os.path.join(flac_dir, filename + ".flac")
            if not os.path.exists(flac_path):
                continue
            (reals if label_str == "bonafide" else fakes).append((flac_path, 0 if label_str == "bonafide" else 1))
    if rng is not None:
        rng.shuffle(reals); rng.shuffle(fakes)
    n = min(max_per_class, len(reals), len(fakes))
    return reals[:n] + fakes[:n], len(reals), len(fakes)


def predict_audio_only(model, mfcc_batch, n_samples=20):
    """MC-Dropout over ONLY the audio branch -- audio_encoder + audio_head.
    Kept separate from mc_logits (Cell 2) because that function also runs
    the visual backbone, which this cross-generator test has no use for
    and no crops to feed it."""
    m = unwrap(model)
    m.eval()
    enable_mc_dropout(m)
    logits = []
    with torch.no_grad():
        for _ in range(n_samples):
            A_enc = m.audio_encoder(mfcc_batch)
            a_rep = masked_mean_std(A_enc, None)
            logits.append(m.audio_head(a_rep).squeeze(-1).float().cpu().numpy())
    m.eval()
    return np.stack(logits, axis=0)   # (n_samples, B)


proto_path, flac_dir = find_asvspoof_eval()
if proto_path is None or flac_dir is None:
    print("[!] ASVspoof 2019 LA eval not found under /kaggle/input (looking for "
          "ASVspoof2019.LA.cm.eval.trl.txt and an ASVspoof2019_LA_eval/flac directory). "
          "Attach the ASVspoof 2019 LA Kaggle dataset and re-run this cell. "
          "Skipping the cross-generator audio test.")
    asv_results = None
else:
    print(f"ASVspoof 2019 LA protocol: {proto_path}\nASVspoof 2019 LA flac dir: {flac_dir}")
    rng_asv = make_rng("asvspoof")
    asv_entries, n_real_total, n_fake_total = load_asvspoof_eval_list(
        proto_path, flac_dir, max_per_class=1000, rng=rng_asv)
    print(f"Eval partition: {n_real_total} bonafide | {n_fake_total} spoof total; "
          f"sampled {len(asv_entries)} ({sum(1 for _, l in asv_entries if l == 0)} real / "
          f"{sum(1 for _, l in asv_entries if l == 1)} fake)")

    m_eval = unwrap(eval_model)
    m_eval.eval()
    asv_true, asv_mean_logit = [], []
    n_skipped = 0
    BATCH = 32
    for i in tqdm(range(0, len(asv_entries), BATCH), desc="Scoring ASVspoof (audio-only, MC-Dropout)"):
        chunk = asv_entries[i:i + BATCH]
        mfccs, chunk_labels = [], []
        for path, lbl in chunk:
            # A single corrupt/unreadable flac out of ~2000 files must not
            # kill an hour-long scoring run -- skip and count it, the same
            # per-file failure tolerance Cell 3's process_job already has
            # for training extraction.
            try:
                y, sr = ffmpeg_extract_wav(path)
                mfccs.append(maybe_cmvn(mfcc_from_waveform(y, sr), use_cmvn=CONFIG["USE_MFCC_CMVN"]))
                chunk_labels.append(lbl)
            except Exception as e:
                n_skipped += 1
        if not mfccs:
            continue
        mfcc_batch = torch.tensor(np.stack(mfccs), dtype=torch.float32).to(device)
        logits = predict_audio_only(m_eval, mfcc_batch, n_samples=CONFIG["N_MC_SAMPLES"])
        asv_mean_logit.append(logits.mean(axis=0))
        asv_true.extend(chunk_labels)
    if n_skipped:
        print(f"  [!] Skipped {n_skipped} unreadable ASVspoof file(s) out of {len(asv_entries)}.")

    asv_true = np.array(asv_true, dtype=int)
    asv_mean_logit = np.concatenate(asv_mean_logit)
    asv_prob = sigmoid(asv_mean_logit / temps["audio"])

    auc_mean, auc_lo, auc_hi = bootstrap_ci(asv_true, asv_prob, roc_auc_score)
    pred_transferred = (asv_prob >= thrs["audio"]).astype(int)
    acc_t = accuracy_score(asv_true, pred_transferred)
    bacc_t = balanced_accuracy_score(asv_true, pred_transferred)
    ece_asv, _, _ = expected_calibration_error(asv_prob, asv_true)
    val_audio_auc = aucs_all["audio"][0]

    print("\n" + "=" * 68)
    print(f"  ASVSPOOF 2019 LA CROSS-GENERATOR AUDIO REPORT (n={len(asv_true)}, "
          f"thr={thrs['audio']:.4f})")
    print("=" * 68)
    print(classification_report(asv_true, pred_transferred, target_names=["Bonafide", "Spoof"],
                                zero_division=0))
    print(f">> Cross-generator AUC (bootstrap 95% CI, THRESHOLD-FREE -- lead with this): "
          f"{auc_mean:.4f} [{auc_lo:.4f}, {auc_hi:.4f}]")
    print(f">> Accuracy at FakeAVCeleb-transferred threshold: {acc_t:.4f} | "
          f"Balanced accuracy: {bacc_t:.4f}")
    print(f">> ECE (post-temperature): {ece_asv:.4f}")
    print(f">> In-dataset (FakeAVCeleb) val audio AUC was {val_audio_auc:.4f}; "
          f"cross-generator drop: {val_audio_auc - auc_mean:+.4f}")
    print("[NOTE] ASVspoof is studio-condition speech vs FakeAVCeleb's YouTube audio -- this AUC "
          "conflates generator shift with domain shift. State that explicitly wherever this number "
          "is reported; treat it as an upper bound on transfer failure, not a clean generator-only "
          "ablation.")
    if auc_lo <= 0.5:
        print("[!] CI includes chance -- the audio head's 0.997 does not transfer to an unseen "
              "generator at all. This is itself a legitimate, publishable negative finding: report "
              "it as evidence the head learned a generator fingerprint, not 'is this speech "
              "synthetic' in general.")
    else:
        print(f">> AUC clears chance -- some synthetic-speech signal generalizes beyond the "
              f"training generator, though {'not' if auc_mean < 0.75 else 'reasonably'} well.")

    fpr_a, tpr_a, _ = roc_curve(asv_true, asv_prob)
    plt.figure(figsize=(6.5, 5.5))
    plt.plot(fpr_a, tpr_a, color='#e377c2', lw=3, label=f'ASVspoof LA (AUC={auc_mean:.3f})')
    plt.plot([0, 1], [0, 1], color='gray', lw=1.2, linestyle='--')
    plt.xlabel('FPR'); plt.ylabel('TPR')
    plt.title(f'Audio Head -- Cross-Generator Transfer\nAUC 95% CI=[{auc_lo:.3f}, {auc_hi:.3f}]',
             fontsize=12, fontweight='bold')
    plt.legend(loc="lower right"); plt.tight_layout()
    plt.savefig("/kaggle/working/crossfuse_v4_asvspoof_roc.png", dpi=300, bbox_inches='tight')
    plt.show()

    asv_results = {
        "n": len(asv_true), "n_real": int((asv_true == 0).sum()), "n_fake": int((asv_true == 1).sum()),
        "auc_mean": auc_mean, "auc_ci": [auc_lo, auc_hi], "accuracy_transferred": acc_t,
        "balanced_accuracy_transferred": bacc_t, "ece": ece_asv, "val_audio_auc": val_audio_auc,
        "cross_generator_drop": val_audio_auc - auc_mean,
        "confound_note": "ASVspoof is studio-condition speech; FakeAVCeleb audio is YouTube-sourced. "
                        "This AUC conflates generator shift with recording-domain shift.",
    }
    with open("/kaggle/working/asvspoof_results_v4.json", "w") as f:
        json.dump(asv_results, f, indent=2)
    print("\nasvspoof_results_v4.json written.")

### Cell 7 — Interactive Inference: Fusion Verdict + Attribution + Margin-to-Threshold
The `fusion_head` (not hand-written if/elif logic over the two attribution heads, as in v3.3) is the
headline verdict, with video/audio kept as attribution signals. Every probability is printed alongside
its MARGIN TO THRESHOLD — v3.3's own in-the-wild run called a clip "DEEPFAKE" at 69.00% against a
67.9% threshold, an 0.011 margin dressed up as a confident verdict. The sync signal is shown only if
`calibration.json` marked it reportable (Cell 5's `SYNC_MIN_AUC` gate). Retains v3.3's real fixes:
`ffmpeg_extract_wav` as the one audio decode path, calibration loaded from `calibration.json` rather
than hardcoded, and the explicit-else handling for the FileUpload `.value` shape that once left
`content` undefined.

In [ ]:
# =====================================================================
# CELL 7: INTERACTIVE IN-THE-WILD INFERENCE -- FUSION VERDICT +
#         PER-MODALITY ATTRIBUTION + GATED AV-SYNC + MARGIN-TO-THRESHOLD
#
# WHAT CHANGED vs v3.3:
#
#  * FUSION HEAD IS THE HEADLINE VERDICT, video/audio remain attribution
#    signals alongside it. v3.3 built its final verdict as hand-written
#    if/elif logic over the two attribution heads plus a sync escalation;
#    that logic could never actually reflect "cross-attention changed the
#    decision" because cross-attention never touched it. The fusion head
#    (Cells 2/4/5) is trained on exactly that fused representation, so
#    using it as the headline signal is what makes the verdict here
#    consistent with what ablation A9 (Cell 8) measures.
#
#  * MARGIN-TO-THRESHOLD IS PRINTED FOR EVERY HEAD. v3.3's own in-the-wild
#    run called a clip "DEEPFAKE" at P(video fake)=69.00% against a
#    67.9% threshold -- an 0.011 margin dressed up as a confident verdict.
#    Every probability below is now printed alongside `margin = p - thr`
#    so a hair's-breadth crossing cannot look identical to a decisive one.
#
#  * SYNC IS GATED ON calibration.json's "sync_reportable" flag (Cell 5,
#    itself gated on the preregistered CONFIG["SYNC_MIN_AUC"]). If the
#    sync head did not clear that bar on TEST, it is not shown here at
#    all -- v3.3 always printed P(AV mismatch) even though its own sync
#    head scored 0.4917 test AUC, worse than a coin flip.
#
#  * CROP PIPELINE, not embeddings: extract_face_crops_contiguous() (Cell
#    1) + the SAME MFCC/log-mel front-ends as training, so an uploaded
#    clip is processed identically to a training/test sample.
#
# RETAINED from v3.3: ffmpeg_extract_wav as the one audio decode path (the
# original fix for the false-positive-audio-via-audioread-fallback bug),
# calibration loaded from calibration.json rather than hardcoded, MTCNN +
# model cached at module scope, and the explicit-else fix for the
# FileUpload `.value` shape that left `content` undefined in v3.2.
# =====================================================================

import os
import json
import numpy as np
import torch
import ipywidgets as widgets
from IPython.display import display, clear_output

with open("/kaggle/working/calibration.json") as f:
    _calib = json.load(f)
INFER_TEMPS = _calib["temp"]
INFER_THRS = _calib["thr"]
INFER_UNC_FLAG = _calib["unc_flag"]
SYNC_REPORTABLE = _calib["sync_reportable"]

print(f"Loaded calibration -> thr={ {h: round(v, 4) for h, v in INFER_THRS.items()} } | "
      f"unc_flag={ {h: round(v, 4) for h, v in INFER_UNC_FLAG.items()} } | "
      f"sync_reportable={SYNC_REPORTABLE} (test AUC {_calib['test_auc'].get('sync', float('nan')):.4f} "
      f"vs SYNC_MIN_AUC {_calib['sync_min_auc']})")

# ---- Module-scope cached detector + model (built once, reused per upload) ----
_infer_mtcnn = build_detector(str(device))
_infer_model = build_model(CONFIG).to(device)
_infer_model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=False))
_infer_model.eval()


def analyze_custom_video(video_path: str):
    print("\n" + "=" * 68)
    print("        CROSSFUSE v4 IN-THE-WILD FORENSIC INFERENCE ENGINE")
    print("=" * 68)

    if not os.path.exists(video_path):
        raise FileNotFoundError(f"ERROR: Could not find video at '{video_path}'.")

    print(f"[*] Target Video      : {os.path.basename(video_path)}")
    print(f"[*] Hardware Device   : {device.type.upper()}")

    print("[*] Extracting a contiguous face-crop window (fine-tuned backbone)...")
    crops, valid, fps, start, ts = extract_face_crops_contiguous(
        video_path, _infer_mtcnn, rng=make_rng(f"infer:{os.path.basename(video_path)}"))
    if crops is None or valid.sum() == 0:
        raise RuntimeError("ERROR: Could not extract any valid face crops from the video!")

    W = CONFIG["WINDOW_FRAMES"]
    T = crops.shape[0]
    w = min(W, T)
    win_start = max(0, T - w) // 2
    window = crops[win_start:win_start + w]
    win_valid = valid[win_start:win_start + w]
    fps_v = fps if fps > 1 else 25.0
    t0 = float(ts[win_start])
    t1 = t0 + w / fps_v
    if w < W:
        pad = W - w
        window = np.concatenate([window, np.zeros((pad, *window.shape[1:]), np.uint8)], 0)
        win_valid = np.concatenate([win_valid, np.zeros(pad, bool)])
    if not win_valid.any():
        win_valid[0] = True

    print("[*] Decoding audio (ffmpeg front-end -- the fix for the earlier false-positive-via-"
          "audioread-fallback bug) and computing MFCC + window-aligned log-mel...")
    y, sr = ffmpeg_extract_wav(video_path)
    mfcc = maybe_cmvn(mfcc_from_waveform(y, sr), use_cmvn=CONFIG["USE_MFCC_CMVN"])
    sync_a = logmel_window(y, sr, t0, t1) if CONFIG["SYNC_DENSE_ALIGNED"] else mfcc.copy()

    print(f"[*] Running fusion + per-modality inference with {CONFIG['N_MC_SAMPLES']}-sample "
          f"MC-Dropout uncertainty...")
    crops_t = torch.from_numpy(np.ascontiguousarray(window)).unsqueeze(0).to(device)
    mfcc_t = torch.from_numpy(mfcc).unsqueeze(0).to(device)
    sync_t = torch.from_numpy(sync_a).unsqueeze(0).to(device)
    mask_t = torch.from_numpy(win_valid).unsqueeze(0).to(device)

    mc = mc_logits(_infer_model, crops_t, mfcc_t, sync_audio=sync_t, visual_mask=mask_t,
                   n_samples=CONFIG["N_MC_SAMPLES"])

    def head_stats(name):
        if name not in mc:
            return None
        logit = mc[name].mean(axis=0)[0]
        std = sigmoid(mc[name]).std(axis=0)[0]
        p = float(sigmoid(logit / INFER_TEMPS[name]))
        thr = INFER_THRS[name]
        return {"p": p, "unc": float(std), "thr": thr, "margin": p - thr,
               "uncertain": std >= INFER_UNC_FLAG.get(name, np.inf), "is_positive": p >= thr}

    video = head_stats("video")
    audio = head_stats("audio")
    fusion = head_stats("fusion")
    sync = head_stats("sync") if SYNC_REPORTABLE else None

    print("\n" + "-" * 68)
    print("                     FORENSIC ANALYSIS REPORT (v4)")
    print("-" * 68)
    for label, h in (("P(video fake)", video), ("P(audio fake)", audio),
                    ("P(overall fake, fusion head)", fusion), ("P(AV mismatch)", sync)):
        if h is None:
            continue
        flag = "  [UNCERTAIN -- flag for review]" if h["uncertain"] else ""
        print(f"  > {label:<32}: {h['p']*100:6.2f}%  (thr {h['thr']*100:5.1f}%, "
              f"margin {h['margin']*100:+6.2f}pp, uncertainty {h['unc']:.4f}){flag}")
    if not SYNC_REPORTABLE:
        print(f"  > P(AV mismatch)                : not shown -- sync head did not clear the "
              f"preregistered SYNC_MIN_AUC ({_calib['sync_min_auc']}) on test.")
    print("-" * 68)

    any_uncertain = any(h is not None and h["uncertain"] for h in (video, audio, fusion))
    if any_uncertain:
        which = [n for n, h in (("video", video), ("audio", audio), ("fusion", fusion))
                if h is not None and h["uncertain"]]
        verdict = f"UNCERTAIN ({'/'.join(which)}) -- flagged for human review"
    elif fusion is None:
        # Defensive fallback: only reachable if CONFIG["USE_FUSION_HEAD"] was
        # manually disabled before training (not the standard main-run
        # config) -- fall back to OR-of-attribution rather than crashing on
        # fusion["is_positive"].
        if video["is_positive"] or audio["is_positive"]:
            verdict = "DEEPFAKE -- attribution head(s) fired (no fusion head trained for this checkpoint)"
        else:
            verdict = "AUTHENTIC MEDIA (no fusion head trained for this checkpoint)"
    elif fusion["is_positive"]:
        detail = []
        if video["is_positive"]:
            detail.append("video")
        if audio["is_positive"]:
            detail.append("audio")
        attribution = f" (attributed to: {'+'.join(detail)})" if detail else " (attribution unclear)"
        verdict = f"DEEPFAKE -- fusion head flags manipulation{attribution}"
    elif sync is not None and sync["is_positive"]:
        verdict = "SUSPICIOUS -- AV streams do not correspond, though neither attribution head fired"
    else:
        verdict = "AUTHENTIC MEDIA -- fusion head finds no manipulation"

    print(f"  > VIDEO attribution  : {'FAKE' if video['is_positive'] else 'REAL'} "
          f"(margin {video['margin']*100:+.2f}pp)")
    print(f"  > AUDIO attribution  : {'FAKE' if audio['is_positive'] else 'REAL'} "
          f"(margin {audio['margin']*100:+.2f}pp)")
    print(f"  > FINAL VERDICT      : {verdict}")
    print("=" * 68 + "\n")

    if fusion is not None and abs(fusion["margin"]) < 0.05:
        print("[!] The fusion-head margin is under 5 percentage points -- treat this call as "
              "borderline, not decisive, regardless of which side of the threshold it landed on.")


upload_widget = widgets.FileUpload(accept='.mp4,.avi,.mov', multiple=False,
                                   description='Upload Video', button_style='primary')
output_area = widgets.Output()


def on_file_upload(change):
    with output_area:
        clear_output()
        if not upload_widget.value:
            return
        print("Processing uploaded file...")
        uploaded_data = upload_widget.value
        file_name, content = None, None
        if isinstance(uploaded_data, (tuple, list)):
            file_info = uploaded_data[0]
            file_name = file_info['name']
            content = file_info['content']
        elif isinstance(uploaded_data, dict):
            file_name = list(uploaded_data.keys())[0]
            content = uploaded_data[file_name]['content']
        else:
            # Retained v3.3 fix: leaving `content` undefined here (as v3.2
            # did) NameErrors on write -- this is the shape some
            # ipywidgets versions' FileUpload trait actually returns.
            print(f"[!] ERROR: unrecognized upload_widget.value type: {type(uploaded_data)}. "
                  f"Cannot extract file content.")
            return
        if file_name is None or content is None:
            print("[!] ERROR: could not parse uploaded file name/content.")
            return

        save_path = os.path.join("/kaggle/working", f"interactive_{file_name}")
        if isinstance(content, memoryview):
            content = content.tobytes()
        with open(save_path, "wb") as f:
            f.write(content)
        print(f"File saved cleanly to: {save_path}")
        analyze_custom_video(save_path)


upload_widget.observe(on_file_upload, names='value')
print("Click the button below to upload your test video directly:")
display(widgets.VBox([upload_widget, output_area]))

### Cell 8 — Ablation Grid (A1–A10)
Reuses cached crops and `run_training_pipeline()` to train nine additional configurations at a
reduced epoch budget (A4 is Cell 4's main run, reused, not retrained) across 2 seeds each. Every row
now shares identical split fractions/sizes (fixing v3.3's group-count-split incomparability), and
`combined_acc` is replaced by balanced accuracy at a per-row, val-fit threshold. **A9 vs. A4 is the
title-deciding row**: an equal-capacity late-fusion MLP swapped in for the CA-fused representation,
everything else held fixed. **A10 vs. A4** isolates whether temporal alignment — not architecture —
was what kept v3.3's sync head at chance. **A8 vs. A4** now isolates PURE identity leakage (v3.3's A8
confounded this with pseudo-replicated near-duplicates crossing the split boundary). Ends by applying
the preregistered title-decision rule from the v4 plan.

In [ ]:
# =====================================================================
# CELL 8: ABLATION GRID (A1-A10)
# Each row reuses the cached crops (Cell 3) and run_training_pipeline()
# (Cell 4) with a modified CONFIG copy -- only the loss composition /
# dataset filter / split strategy / fusion mechanism changes per row.
# A4 IS the Cell 4 main run, reused rather than retrained.
#
# THE BUG THIS CELL'S PREDECESSOR HAD, AND WHY IT NO LONGER CAN:
# v3.3's run_training_pipeline saved the best checkpoint to disk but
# RETURNED the model in its final-epoch state; its quick_test_eval scored
# that returned object. Every v3.3 ablation row was therefore evaluated
# ~7 epochs past its own best checkpoint (this is why A2_audio_only showed
# 0.6265 audio AUC for a head that reaches 0.997 elsewhere). Cell 4's
# run_training_pipeline now reloads the best checkpoint before returning
# -- ONE FIX, applied centrally, so every caller (Cell 5, Cell 6, this
# cell) automatically evaluates the right weights. Nothing here re-does
# that fix; the regression test for it is A2's audio AUC below (expect
# ~0.99, not ~0.63).
#
# WHAT ELSE CHANGED vs v3.3's ablation grid:
#  * IDENTICAL SPLIT FRACTIONS AND SIZES for every row. v3.3's
#    GroupShuffleSplit split by GROUP count, not sample count, so its main
#    run's test set (946) and A8's (750) were different sizes -- rows were
#    not comparable. build_3way_split_packed (Cell 4) targets the same
#    SPLIT_FRACS for every row regardless of identity_disjoint, so A8 vs
#    A4 now isolates identity leakage alone.
#  * combined_acc (hard 0.5 threshold on uncalibrated OR-of-labels) is
#    REPLACED by balanced accuracy at a per-row, val-fit threshold --
#    meaningless swings from an arbitrary cutoff are gone.
#  * 2 SEEDS PER ROW, reported as mean +/- std, not a single run.
#  * A3 no longer means "OR-rule late fusion" (that was a hand-written
#    rule, not a trained baseline) -- it is now "two independent heads,
#    no fusion mechanism at all" (USE_FUSION_HEAD=False), a genuinely
#    distinct question from A9.
#  * A9 is NEW and IS THE TITLE-DECIDING ROW: LATE_FUSION_BASELINE=True
#    keeps cross-attention (and the sync head) fully intact and trains an
#    EQUAL-CAPACITY MLP over concat[v_rep, a_rep] instead of the CA-fused
#    representation for the fusion_head ONLY -- the one thing that
#    changes is whether the fusion decision passed through cross-
#    attention. v3.4's A9 compared against max() of two temperature-
#    scaled head probabilities, which is not invariant to per-head
#    temperature; that "null result" was a comparison artefact, not a
#    finding. Compare A9 to A4 by AUC (threshold-free).
#  * A10 is NEW: SYNC_DENSE_ALIGNED=False feeds the sync head the
#    v3.3-style whole-clip-strided/unaligned input through the IDENTICAL
#    architecture, isolating whether TEMPORAL ALIGNMENT (not the
#    architecture) was what kept v3.3's sync head at chance.
#
# Metrics here are QUICK -- a single deterministic (dropout-off) forward
# pass, a fast Youden threshold fit on val, no MC-Dropout, no bootstrap CI,
# no temperature scaling. Adequate for RANKING configurations against each
# other; not a substitute for Cell 5/6's calibrated, bootstrapped numbers
# on the main run.
# =====================================================================

import copy
import csv
import json
import numpy as np
import torch
from collections import defaultdict
from sklearn.metrics import roc_curve, balanced_accuracy_score, accuracy_score

N_ABLATION_SEEDS = 2
ABLATION_EPOCHS = dict(EPOCHS_FROZEN=3, EPOCHS_FINETUNE=12, PATIENCE=5)


def cfg_variant(**overrides):
    c = copy.deepcopy(CONFIG)
    c.update(ABLATION_EPOCHS)
    c.update(overrides)
    return c


def quick_youden(y_true, y_prob, default=0.5):
    y_true = np.asarray(y_true)
    if len(set(y_true.astype(int).tolist())) < 2:
        return default
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    return float(thr[np.argmax(tpr - fpr)])


def quick_eval_row(result, cfg):
    """One deterministic pass on val (to fit a quick per-row threshold)
    and one on test (to report AUC + balanced accuracy at that threshold)
    -- see header for why this replaces v3.3's combined_acc."""
    val = evaluate_deterministic(result["model"], result["val_loader"], cfg)
    test = evaluate_deterministic(result["model"], result["test_loader"], cfg)
    row = {"val_video_auc": val["video_auc"], "val_audio_auc": val["audio_auc"],
          "val_sync_auc": val["sync_auc"], "val_fusion_auc": val["fusion_auc"],
          "test_video_auc": test["video_auc"], "test_audio_auc": test["audio_auc"],
          "test_sync_auc": test["sync_auc"], "test_fusion_auc": test["fusion_auc"],
          "n_test": len(result["test_idx"])}
    for head, vkey, tkey in (("video", "v", "v"), ("audio", "a", "a"),
                            ("fusion", "f", "f"), ("sync", "s", "s")):
        vt, vp = val["raw"].get(f"{vkey}_true"), val["raw"].get(f"{vkey}_prob")
        tt, tp = test["raw"].get(f"{tkey}_true"), test["raw"].get(f"{tkey}_prob")
        if not vt or not tt:
            row[f"{head}_bacc"] = float('nan')
            continue
        thr = quick_youden(vt, vp)
        pred = (np.asarray(tp) >= thr).astype(int)
        row[f"{head}_bacc"] = (balanced_accuracy_score(tt, pred)
                               if len(set(np.asarray(tt).astype(int).tolist())) > 1 else float('nan'))

    # A3-style "no trained fusion" rows still get a reportable overall
    # number: OR of the two attribution heads' OWN val-fit thresholds,
    # evaluated on test. This is descriptive (what would naive fusion look
    # like), never trained, and never claimed as a fusion mechanism.
    if not cfg["USE_FUSION_HEAD"]:
        vt, vp = val["raw"]["v_true"], val["raw"]["v_prob"]
        at, ap = val["raw"]["a_true"], val["raw"]["a_prob"]
        thr_v, thr_a = quick_youden(vt, vp), quick_youden(at, ap)
        tvt, tvp = test["raw"]["v_true"], test["raw"]["v_prob"]
        tat, tap = test["raw"]["a_true"], test["raw"]["a_prob"]
        or_pred = ((np.asarray(tvp) >= thr_v) | (np.asarray(tap) >= thr_a)).astype(int)
        or_true = np.clip(np.asarray(tvt) + np.asarray(tat), 0, 1).astype(int)
        row["or_rule_bacc"] = (balanced_accuracy_score(or_true, or_pred)
                               if len(set(or_true.tolist())) > 1 else float('nan'))
    return row


ABLATIONS = [
    ("A1_video_only", cfg_variant(MODALITY="video_only", USE_CROSS_ATTENTION=False,
                                  USE_SYNC_HEAD=False, USE_FUSION_HEAD=False)),
    ("A2_audio_only", cfg_variant(MODALITY="audio_only", USE_CROSS_ATTENTION=False,
                                  USE_SYNC_HEAD=False, USE_FUSION_HEAD=False)),
    ("A3_two_head_no_fusion", cfg_variant(USE_CROSS_ATTENTION=False, USE_SYNC_HEAD=False,
                                          USE_FUSION_HEAD=False)),
    # A4 = Cell 4's main run, reused below at seed=CONFIG["SEED"] (no retrain).
    ("A5_no_hard_negatives", cfg_variant(GENERATE_HARD_NEGATIVES=False)),
    ("A6_no_compression_aug", cfg_variant(USE_COMPRESSION_AUG=False)),
    ("A7_no_mfcc_cmvn", cfg_variant(USE_MFCC_CMVN=False)),
    ("A8_random_split", cfg_variant(IDENTITY_DISJOINT_SPLIT=False)),
    ("A9_late_fusion_baseline", cfg_variant(LATE_FUSION_BASELINE=True)),
    ("A10_sync_unaligned", cfg_variant(SYNC_DENSE_ALIGNED=False)),
]

print(f"Running {len(ABLATIONS)} ablation configurations x {N_ABLATION_SEEDS} seeds "
      f"(A4 reuses Cell 4's main run, seed {CONFIG['SEED']}, at full epoch budget)...\n")

ablation_rows = []

# ---- A4: reuse the already-trained main-run seed, zero extra cost -------
# NOTE: Cell 4 only keeps the LAST seed's model resident (earlier seeds'
# "model" entries were freed to fit 3 fine-tuned backbones in GPU memory
# across the main multi-seed run), so A4 must reuse _main_result (the same
# checkpoint Cell 5/6 calibrate and report on) rather than seed_results[0].
a4_row = quick_eval_row(_main_result, _main_result["cfg"])
a4_row.update(row="A4_full_crossfuse", seed=_main_result["seed"])
ablation_rows.append(a4_row)
print(f"A4_full_crossfuse (seed {a4_row['seed']}, = main run) -> "
     f"video={a4_row['test_video_auc']:.4f} audio={a4_row['test_audio_auc']:.4f} "
     f"fusion={a4_row['test_fusion_auc']:.4f} sync={a4_row['test_sync_auc']:.4f}")

ablation_csv_path = "/kaggle/working/ablation_results_v4.csv"
fieldnames = ["row", "seed", "n_test", "val_video_auc", "val_audio_auc", "val_sync_auc",
             "val_fusion_auc", "test_video_auc", "test_audio_auc", "test_sync_auc",
             "test_fusion_auc", "video_bacc", "audio_bacc", "fusion_bacc", "sync_bacc", "or_rule_bacc"]


def _write_ablation_csv():
    """Rewrite the (small, ~18-row) CSV after every row completes -- so a
    crash partway through 9 configs x 2 seeds of fine-tuning loses at most
    the row in progress, not every row already finished."""
    with open(ablation_csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(ablation_rows)


_write_ablation_csv()   # header + A4 row, so the file exists even if row 1 below fails

failed_rows = []
for tag, cfg in ABLATIONS:
    for si in range(N_ABLATION_SEEDS):
        seed = CONFIG["SEED"] + si
        try:
            result = run_training_pipeline(cfg, tag=tag, seed=seed, verbose=False)
            row = quick_eval_row(result, cfg)
            row.update(row=tag, seed=seed)
            ablation_rows.append(row)
            print(f"{tag} (seed {seed}) -> video={row['test_video_auc']:.4f} "
                 f"audio={row['test_audio_auc']:.4f} fusion={row['test_fusion_auc']:.4f} "
                 f"sync={row['test_sync_auc']:.4f}")
            del result
        except Exception as e:
            # One bad row (CUDA OOM, a NaN loss, etc.) must not cost the
            # other 17 rows' worth of already-completed training. Log it,
            # keep going, and it shows up in the summary as a gap to
            # re-run rather than a lost session.
            print(f"[!] {tag} (seed {seed}) FAILED: {type(e).__name__}: {e} -- skipping this row.")
            failed_rows.append((tag, seed, f"{type(e).__name__}: {e}"))
        torch.cuda.empty_cache()
        _write_ablation_csv()

print(f"\nPer-seed rows written -> {ablation_csv_path}")
if failed_rows:
    print(f"[!] {len(failed_rows)} row(s) failed and are MISSING from the summary below -- "
         f"re-run them individually: {failed_rows}")

# ---- Aggregate to mean +/- std per row -----------------------------------
by_row = defaultdict(list)
for r in ablation_rows:
    by_row[r["row"]].append(r)

print("\n" + "=" * 78)
print("  ABLATION SUMMARY (mean +/- std across seeds; A4 is the single main-run seed)")
print("=" * 78)
summary_rows = []
metric_cols = ["test_video_auc", "test_audio_auc", "test_fusion_auc", "test_sync_auc"]
for tag in ["A4_full_crossfuse"] + [t for t, _ in ABLATIONS]:
    rows = by_row[tag]
    line = {"row": tag, "n_seeds": len(rows)}
    parts = [f"{tag:<26}"]
    for col in metric_cols:
        vals = [r[col] for r in rows if not np.isnan(r[col])]
        m, s = (np.mean(vals), np.std(vals)) if vals else (float('nan'), float('nan'))
        line[col + "_mean"], line[col + "_std"] = float(m), float(s)
        short = col.replace("test_", "").replace("_auc", "")
        parts.append(f"{short}={m:.4f}+/-{s:.4f}")
    summary_rows.append(line)
    print("  " + "  ".join(parts))

print("\nKey comparisons for the paper:")
print("  A3 (no fusion mechanism) vs A4/A9 (any trained fusion) -> does explicit fusion help at all.")
print("  A9 (late-fusion MLP, CA elsewhere unchanged) vs A4 (CA-fused verdict) -> TITLE-DECIDING: "
     "isolates whether cross-attention itself improves the fusion decision.")
print("  A10 (unaligned sync input) vs A4 (dense, time-aligned) -> isolates whether TEMPORAL "
     "ALIGNMENT, not architecture, was what kept v3.3's sync head at chance.")
print("  A8 (random split) vs A4 (identity-disjoint) -> now isolates PURE identity leakage "
     "(v3.3's A8 confounded this with pseudo-replicated near-duplicates crossing the split boundary).")

with open("/kaggle/working/ablation_summary_v4.json", "w") as f:
    json.dump(summary_rows, f, indent=2)
print("\nablation_summary_v4.json written.")

# ---- Title decision rule (preregistered in the v4 plan) ------------------
def _row_mean(tag, col):
    vals = [r[col] for r in by_row.get(tag, []) if not np.isnan(r[col])]
    return float(np.mean(vals)) if vals else float('nan')

a4_fusion_auc = _row_mean("A4_full_crossfuse", "test_fusion_auc")
a9_fusion_auc = _row_mean("A9_late_fusion_baseline", "test_fusion_auc")
a9_std = np.std([r["test_fusion_auc"] for r in by_row.get("A9_late_fusion_baseline", [])
                if not np.isnan(r["test_fusion_auc"])]) if by_row.get("A9_late_fusion_baseline") else float('nan')
a4_sync_auc = a4_row["test_sync_auc"]
a10_sync_auc = _row_mean("A10_sync_unaligned", "test_sync_auc")

print("\n" + "=" * 78)
print("  TITLE DECISION (preregistered rule from the v4 plan)")
print("=" * 78)
print(f"  A4 fusion (CA) AUC:        {a4_fusion_auc:.4f}")
print(f"  A9 fusion (late, no CA):   {a9_fusion_auc:.4f} +/- {a9_std:.4f}")
print(f"  A4 sync AUC:               {a4_sync_auc:.4f}  (SYNC_MIN_AUC = {CONFIG['SYNC_MIN_AUC']})")
print(f"  A10 sync (unaligned) AUC:  {a10_sync_auc:.4f}")

ca_beats_late_fusion = (not np.isnan(a9_std)) and (a4_fusion_auc - a9_fusion_auc > a9_std)
sync_clears_bar = a4_sync_auc >= CONFIG["SYNC_MIN_AUC"]

if ca_beats_late_fusion:
    print("\n  => A9: CA-fused verdict beats late fusion by more than the seed std. "
         "The 'Cross-Attention Fusion' title clause is EARNED.")
elif sync_clears_bar:
    print("\n  => A10/Cell 5: sync AUC clears SYNC_MIN_AUC. Cross-attention is load-bearing for a "
         "second, independent reason. KEEP the title regardless of A9.")
else:
    print("\n  => BOTH null. Recommendation: drop the cross-attention claim from the title; lead "
         "with identity-leakage quantification (A8), per-modality attribution, calibration, and "
         "the two cross-domain transfer results (Cells 6/6b). Keep 'CrossFuse' as the system name; "
         "cross-attention becomes a described component, not a headline claim.")

### Cell 9 — Paper Artifact Export
Collects every JSON/CSV this notebook produced into `results_v4.json` and generates ready-to-paste
LaTeX for the multi-seed validation, calibrated test, cross-dataset (Celeb-DF + DFDC appendix),
cross-generator-audio (ASVspoof), and ablation tables. All figures were already saved at 300 dpi in
Cells 5/6/6b.

In [ ]:
# =====================================================================
# CELL 9: PAPER ARTIFACT EXPORT (v4)
# Collects every JSON/CSV this notebook wrote into results_v4.json, and
# generates ready-to-paste LaTeX for the validation, cross-dataset
# (Celeb-DF v2 + DFDC appendix), cross-generator-audio (ASVspoof), and
# ablation tables. All figures were already saved at 300 dpi in
# Cells 5/6/6b.
# =====================================================================

import json
import os
import csv
from collections import defaultdict


def _load_json(path, default=None):
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    print(f"[!] Missing: {path} (run the corresponding cell first)")
    return default


def _load_csv(path):
    if os.path.exists(path):
        with open(path, newline="") as f:
            return list(csv.DictReader(f))
    print(f"[!] Missing: {path} (run Cell 8 first)")
    return []


best_metrics_j = _load_json("/kaggle/working/best_metrics_v4.json", {})
calibration_j = _load_json("/kaggle/working/calibration.json", {})
celebdf_j = _load_json("/kaggle/working/celebdf_results_v4.json", {})
dfdc_j = _load_json("/kaggle/working/dfdc_results_v4.json", {})
asvspoof_j = _load_json("/kaggle/working/asvspoof_results_v4.json", {})
ablation_rows_j = _load_csv("/kaggle/working/ablation_results_v4.csv")
ablation_summary_j = _load_json("/kaggle/working/ablation_summary_v4.json", [])

results_v4 = {
    "paper_title": "CrossFuse: Multi-Modal Deepfake Detection Using Cross-Attention Fusion of "
                   "Audio and Visual Streams",
    "notebook_version": "v4.0",
    "config": {k: str(v) for k, v in CONFIG.items()},
    "training": best_metrics_j,
    "calibration": calibration_j,
    "celebdf_zero_shot": celebdf_j,
    "dfdc_zero_shot_appendix": dfdc_j,
    "asvspoof_cross_generator_audio": asvspoof_j,
    "ablations_per_seed": ablation_rows_j,
    "ablations_summary": ablation_summary_j,
}
with open("/kaggle/working/results_v4.json", "w") as f:
    json.dump(results_v4, f, indent=2)
print("results_v4.json written.")


def fmt(x, nd=4):
    try:
        return f"{float(x):.{nd}f}"
    except (TypeError, ValueError):
        return "--"


# ---- Table 1: multi-seed validation performance --------------------------
per_seed = best_metrics_j.get("per_seed", [])
seed_lines = "\n".join(
    r"%s & %s & %s & %s & %s & %s \\" % (
        s.get("seed", "--"), s.get("best_epoch", "--"), fmt(s.get("video")), fmt(s.get("audio")),
        fmt(s.get("sync")), fmt(s.get("fusion")))
    for s in per_seed)
vm, vs = best_metrics_j.get("val_video_auc_mean_std", [None, None])
am, a_s = best_metrics_j.get("val_audio_auc_mean_std", [None, None])
sm, ss = best_metrics_j.get("val_sync_auc_mean_std", [None, None])
fm, fs = best_metrics_j.get("val_fusion_auc_mean_std", [None, None])
table1 = r"""
\begin{table}[t]
\centering
\caption{CrossFuse v4 validation AUC across %s seeds (identity-disjoint, sample-proportional
60/20/20 split, checkpoint selected on mean of video+audio AUC).}
\begin{tabular}{lccccc}
\toprule
Seed & Best Epoch & Video & Audio & Sync & Fusion \\
\midrule
%s
\midrule
Mean $\pm$ std & -- & %s $\pm$ %s & %s $\pm$ %s & %s $\pm$ %s & %s $\pm$ %s \\
\bottomrule
\end{tabular}
\end{table}
""" % (best_metrics_j.get("n_seeds", "--"), seed_lines, fmt(vm), fmt(vs), fmt(am), fmt(a_s),
       fmt(sm), fmt(ss), fmt(fm), fmt(fs))

# ---- Table 2: TEST-set report (Cell 5, calibrated, matched MC estimator) --
test_auc = calibration_j.get("test_auc", {})
thr = calibration_j.get("thr", {})
temp = calibration_j.get("temp", {})
sync_reportable = calibration_j.get("sync_reportable", False)
table2_lines = "\n".join(
    r"%s & %s & %s & %s \\" % (h.capitalize(), fmt(test_auc.get(h)), fmt(thr.get(h)), fmt(temp.get(h)))
    for h in ("video", "audio", "fusion", "sync") if h in test_auc)
table2 = r"""
\begin{table}[t]
\centering
\caption{CrossFuse v4 test-set performance (calibrated on val, matched MC-Dropout estimator for
both fitting and reporting -- see Root Cause 2 discussion). Sync head reportable: %s
(SYNC\_MIN\_AUC = %s).}
\begin{tabular}{lccc}
\toprule
Head & Test AUC & Threshold & Temperature \\
\midrule
%s
\bottomrule
\end{tabular}
\end{table}
""" % (sync_reportable, calibration_j.get("sync_min_auc", "--"), table2_lines)

# ---- Table 3: cross-dataset generalization (Celeb-DF headline + DFDC appendix) --
def gen_row(name, j):
    if not j:
        return r"%s & \multicolumn{4}{c}{not run} \\" % name
    ci = j.get("auc_ci", [None, None])
    return r"%s (n=%s) & %s [%s, %s] & %s & %s & %s \\" % (
        name, j.get("n", "--"), fmt(j.get("auc_mean")), fmt(ci[0]), fmt(ci[1]),
        fmt(j.get("accuracy_transferred")), fmt(j.get("ece")), fmt(j.get("generalization_gap")))

table3 = r"""
\begin{table}[t]
\centering
\caption{Zero-shot cross-dataset generalization, video head (bootstrap 95\%% CI). Celeb-DF v2 is
the headline benchmark; DFDC is retained only as a small-n appendix (its CI has historically
included chance).}
\begin{tabular}{lcccc}
\toprule
Dataset & AUC [95\%% CI] & Accuracy & ECE & Gen. Gap \\
\midrule
%s
%s
\bottomrule
\end{tabular}
\end{table}
""" % (gen_row("Celeb-DF v2", celebdf_j), gen_row("DFDC (appendix)", dfdc_j))

# ---- Table 4: cross-generator audio transfer (ASVspoof) -------------------
if asvspoof_j:
    ci = asvspoof_j.get("auc_ci", [None, None])
    table4 = r"""
\begin{table}[t]
\centering
\caption{Cross-generator audio transfer: audio head zero-shot on ASVspoof 2019 LA (n=%s),
vs.\ in-dataset FakeAVCeleb val AUC %s. ASVspoof is studio-condition speech vs.\
FakeAVCeleb's YouTube-sourced audio, so this AUC conflates generator shift with recording-domain
shift; treat as an upper bound on transfer failure.}
\begin{tabular}{lc}
\toprule
Metric & Value \\
\midrule
ASVspoof AUC [95\%% CI] & %s [%s, %s] \\
Accuracy (FakeAVCeleb-transferred thr.) & %s \\
In-dataset val AUC & %s \\
Cross-generator drop & %s \\
\bottomrule
\end{tabular}
\end{table}
""" % (asvspoof_j.get("n", "--"), fmt(asvspoof_j.get("val_audio_auc")), fmt(asvspoof_j.get("auc_mean")),
       fmt(ci[0]), fmt(ci[1]), fmt(asvspoof_j.get("accuracy_transferred")),
       fmt(asvspoof_j.get("val_audio_auc")), fmt(asvspoof_j.get("cross_generator_drop")))
else:
    table4 = r"% ASVspoof cross-generator audio test was not run."

# ---- Table 5: ablation grid (mean +/- std across seeds) -------------------
by_row = defaultdict(list)
for r in ablation_rows_j:
    by_row[r["row"]].append(r)

def mean_std(rows, key):
    vals = [float(r[key]) for r in rows if r.get(key) not in (None, "", "nan")]
    if not vals:
        return float('nan'), float('nan')
    import numpy as _np
    return float(_np.mean(vals)), float(_np.std(vals))

ordered_tags = ["A4_full_crossfuse", "A1_video_only", "A2_audio_only", "A3_two_head_no_fusion",
               "A5_no_hard_negatives", "A6_no_compression_aug", "A7_no_mfcc_cmvn",
               "A8_random_split", "A9_late_fusion_baseline", "A10_sync_unaligned"]
ablation_lines = []
for tag in ordered_tags:
    rows = by_row.get(tag, [])
    if not rows:
        continue
    vm_, vs_ = mean_std(rows, "test_video_auc")
    am_, as_ = mean_std(rows, "test_audio_auc")
    fm_, fs_ = mean_std(rows, "test_fusion_auc")
    sm_, ss_ = mean_std(rows, "test_sync_auc")
    ablation_lines.append(
        r"%s & %s$\pm$%s & %s$\pm$%s & %s$\pm$%s & %s$\pm$%s \\" % (
            tag.replace("_", r"\_"), fmt(vm_, 3), fmt(vs_, 3), fmt(am_, 3), fmt(as_, 3),
            fmt(fm_, 3), fmt(fs_, 3), fmt(sm_, 3), fmt(ss_, 3)))

table5 = r"""
\begin{table}[t]
\centering
\caption{Ablation study (FakeAVCeleb held-out test split, mean $\pm$ std across seeds; A4 is a
single seed reused from the main run). A9 vs.\ A4 is the row that decides the paper's title; A8
vs.\ A4 quantifies pure identity-leakage inflation.}
\begin{tabular}{lcccc}
\toprule
Configuration & Video AUC & Audio AUC & Fusion AUC & Sync AUC \\
\midrule
%s
\bottomrule
\end{tabular}
\end{table}
""" % "\n".join(ablation_lines)

for name, tex in [("table1_val_multiseed", table1), ("table2_test_calibrated", table2),
                  ("table3_cross_dataset", table3), ("table4_cross_generator_audio", table4),
                  ("table5_ablations", table5)]:
    path = f"/kaggle/working/{name}.tex"
    with open(path, "w") as f:
        f.write(tex)
    print(f"Wrote {path}")

print("\n" + "=" * 68)
for label, tex in [("Table 1 (val, multi-seed)", table1), ("Table 2 (test, calibrated)", table2),
                   ("Table 3 (cross-dataset)", table3), ("Table 4 (cross-generator audio)", table4),
                   ("Table 5 (ablations)", table5)]:
    print(f"{label} preview:")
    print(tex)

print("\nAll figures were already saved at 300 dpi:")
for fig in ["crossfuse_v4_perhead_plots.png", "crossfuse_v4_sync_roc.png",
           "crossfuse_v4_attribution_matrix.png", "crossfuse_v4_celebdfv2_roc.png",
           "crossfuse_v4_dfdc_roc.png", "crossfuse_v4_asvspoof_roc.png"]:
    p = f"/kaggle/working/{fig}"
    print(f"  {'[OK]' if os.path.exists(p) else '[MISSING]'} {p}")

print("\nv4 paper artifact export complete.")